<a href="https://colab.research.google.com/github/samer-glitch/TADP-Cluster-Computing/blob/main/08_TADP_Experiment_G_Adversarial_Diagnostic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import files
import io
import os

print("📤 Please upload the diabetes_130US.csv file:")
uploaded = files.upload()
DATA_FILENAME  = "diabetes_130US.csv"
DATA_CACHE_DIR = "./data_cache"
os.makedirs(DATA_CACHE_DIR, exist_ok=True)

# Get the uploaded file
for filename in uploaded.keys():
    file_data = uploaded[filename]
    print(f'✅ Uploaded: {filename} ({len(file_data)} bytes)')


    # Save to cache
    cache_path = os.path.join(DATA_CACHE_DIR, DATA_FILENAME)
    with open(cache_path, 'wb') as f:
        f.write(file_data)
    print(f'✅ Saved to cache: {cache_path}')

    # Verify the file was saved
    if os.path.exists(cache_path):
        file_size = os.path.getsize(cache_path)
        print(f'✅ Verified: {cache_path} exists ({file_size} bytes)')
    else:
        print(f'❌ Error: File was not saved properly')

# ======================================================================================
# TADP v20.1 — 2 scenarios - 3 seeds - TRUSTWORTHY DATA PREPARATION EXPERIMENT CORE
# ======================================================================================
# Design guarantees:
#   1) GLOBAL holdout is created BEFORE client partitioning.
#   2) Only TRAIN is distributed to clients.
#   3) Preprocessing parameters/vocabularies use TRAIN only.
#   4) DQ, GX and TADP governance use TRAIN only.
#   5) Held-out TEST never affects preprocessing, governance, class weights,
#      client selection, model initialization, training, or matched-control budgets.
#   6) TEST is used only after training for final evaluation.
#
# Final governance policy used by this diagnostic:
#   - 28 factors across six dimensions (4, 8, 4, 3, 5, 4).
#   - All 28 factor scores must be present, finite, and strictly >0.
#   - Every dimension score must be >=2.5/5.
#   - HPS <3.0 -> Reject; HPS >=3.5 -> Direct Accept.
#   - 3.0 <= HPS <3.5 -> critical-dimension automated review.
#   - HPS_review uses D1, D2, D5, and D6 with renormalized HPS weights;
#     HPS_review >=3.25 -> Accept, otherwise Reject.
#   - Human reviewers verify documentary evidence only; scoring and admission are automated.
#
# GX Core comparator:
#   - Separate from HPS/TADP.
#   - Uses REAL Great Expectations GX Core validation on TRAIN-only client data.
#   - Uses common technical checks: schema, datatype consistency, required ranges,
#     missingness, duplicate/ID integrity, label/domain validity, and structure.
#   - Uses GX native severity-aware validation: zero critical failures required;
#     no ranking and no forced-K.
#
# Runtime/reporting:
#   - Every FL round prints configuration/run/scenario/round progress plus TRAIN-only
#     diagnostic utility and operational metrics.
#   - Every completed scenario prints all predictive and operational metrics.
#   - Scenario checkpoints support restart/resume.
#   - Final result ZIP downloads automatically in Google Colab.
# ======================================================================================

import os
import sys
import gc
import math
import time
import json
import random
import hashlib
import threading
import zipfile
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


# ======================================================================================
# POLICY CONSTANTS — v20.0
# ======================================================================================

GOOD_CUT = 3.0
HIGH_CUT = 3.5
MAX_FACTOR_SCORE = 5.0
GE_ACCEPT_COUNT = 6

DIMENSION_MIN_FLOOR = 2.5
REVIEW_ACCEPT_THRESHOLD = (GOOD_CUT + HIGH_CUT) / 2.0  # 3.25 on the 0..5 rubric scale

# ----------------------------------------------------------------------
# FINAL A1/A2 ADMISSION POLICY
# ----------------------------------------------------------------------
# Gate 1: all 28 factor scores must be present, finite, and strictly >0.
# Gate 2: every dimension score must be >=2.5/5.
# HPS <3.0 -> reject; HPS >=3.5 -> direct accept.
# Intermediate HPS uses critical-dimension HPS_review over D1,D2,D5,D6.
# HPS_review >=3.25 -> accept; otherwise reject.
CRITICAL_REVIEW_DIMS=("dim1","dim2","dim5","dim6")

WEIGHTS_PSCORE_DEFAULT = {
    "dim1": 0.25,  # Source Reliability
    "dim2": 0.15,  # Data Quality and Health
    "dim3": 0.10,  # Documentation Practices
    "dim4": 0.10,  # Timeliness and Refresh Rate
    "dim5": 0.30,  # Regulatory / Compliance Alignment
    "dim6": 0.10,  # Context / Usage Constraints
}

CRITICAL_REVIEW_WEIGHTS = {
    d: WEIGHTS_PSCORE_DEFAULT[d] / sum(WEIGHTS_PSCORE_DEFAULT[q] for q in CRITICAL_REVIEW_DIMS)
    for d in CRITICAL_REVIEW_DIMS
}
_EXPECTED_CRITICAL_REVIEW_WEIGHTS={"dim1":0.3125,"dim2":0.1875,"dim5":0.375,"dim6":0.125}
assert all(math.isclose(CRITICAL_REVIEW_WEIGHTS[d],v,rel_tol=0.0,abs_tol=1e-12) for d,v in _EXPECTED_CRITICAL_REVIEW_WEIGHTS.items())

DIMENSION_NAMES = {
    "dim1": "Source Reliability",
    "dim2": "Data Quality and Health",
    "dim3": "Documentation Practices",
    "dim4": "Timeliness and Refresh Rate",
    "dim5": "Regulatory and Compliance Alignment",
    "dim6": "Context and Usage Constraints",
}

# Factor-count notes:
#   dim1: data_collection_lineage
#   dim2: structural_constraint_integrity
#
# Total = 4 + 8 + 4 + 3 + 5 + 4 = 28 factors.
FACTOR_NAMES = {
    "dim1": [
        "source_reputation",
        "data_controller",
        "data_objective",
        "data_collection_lineage",
    ],
    "dim2": [
        "completeness",
        "duplication_rate",
        "value_validity_error_rate",
        "type_consistency",
        "label_integrity",
        "feature_distribution_consistency",
        "feature_category_coverage",
        "structural_constraint_integrity",
    ],
    "dim3": [
        "data_dictionary",
        "version_logs",
        "collection_protocol",
        "definition_updates",
    ],
    "dim4": [
        "data_freshness",
        "scheduled_refresh",
        "retention_clarity",
    ],
    "dim5": [
        "regulation_coverage",
        "consent_ethics",
        "geo_restrictions",
        "sensitivity_classification",
        "audits",
    ],
    "dim6": [
        "license_terms",
        "ethical_reviews",
        "redistribution",
        "user_agreements",
    ],
}

DOCUMENTARY_DIMS = ("dim1", "dim3", "dim4", "dim5", "dim6")

# ----------------------------------------------------------------------
# COMPLETE 0--5 RUBRIC DESCRIPTORS
# ----------------------------------------------------------------------
# These descriptors reproduce the Appendix-A semantics and add the two
# two reviewer-driven factors explicitly. Controlled evidence is sampled at the factor
# level; HPS is never generated directly.
RUBRIC_DESCRIPTORS = {
    "dim1": {
        "source_reputation": {
            0: "No info",
            1: "Poor",
            2: "Limited evidence",
            3: "Average, partially trusted",
            4: "Well-documented, reliable",
            5: "Highly reputable, verified",
        },
        "data_controller": {
            0: "No documented controller",
            1: "Unclear",
            2: "Partially clear",
            3: "Moderately clear",
            4: "Mostly clear",
            5: "Fully documented",
        },
        "data_objective": {
            0: "None",
            1: "Vague",
            2: "Partial",
            3: "General but unclear",
            4: "Mostly explicit",
            5: "Fully explicit, justified",
        },
        "data_collection_lineage": {
            0: "Collection origin unknown",
            1: "Informal or unverifiable origin",
            2: "Partially documented acquisition path",
            3: "Documented acquisition with limited traceability",
            4: "Well-documented and traceable acquisition path",
            5: "Fully source-linked, versioned, and auditable lineage",
        },
    },
    "dim2": {
        "completeness": {
            0: ">50% missing",
            1: "20-50% missing",
            2: "10-20% missing",
            3: "5-10% missing",
            4: "1-5% missing",
            5: "<1% missing",
        },
        "duplication_rate": {
            0: ">20% duplicates",
            1: "10-20% duplicates",
            2: "5-10% duplicates",
            3: "2-5% duplicates",
            4: "1-2% duplicates",
            5: "<1% duplicates",
        },
        "value_validity_error_rate": {
            0: ">15% invalid/error values",
            1: "10-15% invalid/error values",
            2: "5-10% invalid/error values",
            3: "2-5% invalid/error values",
            4: "1-2% invalid/error values",
            5: "<1% invalid/error values",
        },
        "type_consistency": {
            0: "Highly inconsistent",
            1: "Frequent type inconsistency",
            2: "Moderate type inconsistency",
            3: "Minor type inconsistency",
            4: "Rare type inconsistency",
            5: "Fully consistent",
        },
        "label_integrity": {
            0: ">10% missing/invalid/known erroneous labels",
            1: "5-10% missing/invalid/known erroneous labels",
            2: "2-5% missing/invalid/known erroneous labels",
            3: "1-2% missing/invalid/known erroneous labels",
            4: "0.1-1% missing/invalid/known erroneous labels",
            5: "<=0.1% missing/invalid/known erroneous labels",
        },
        "feature_distribution_consistency": {
            0: "JSD >0.20",
            1: "JSD 0.10-0.20",
            2: "JSD 0.05-0.10",
            3: "JSD 0.025-0.05",
            4: "JSD 0.01-0.025",
            5: "JSD <=0.01",
        },
        "feature_category_coverage": {
            0: "<50% reference support represented",
            1: "50-65% reference support represented",
            2: "65-75% reference support represented",
            3: "75-82.5% reference support represented",
            4: "82.5-90% reference support represented",
            5: ">=90% reference support represented",
        },
        "structural_constraint_integrity": {
            0: ">10% records/structures violate required constraints",
            1: "5-10% violate required constraints",
            2: "2-5% violate required constraints",
            3: "1-2% violate required constraints",
            4: "0.1-1% violate required constraints",
            5: "<=0.1% violate required constraints",
        },
    },
    "dim3": {
        "data_dictionary": {
            0: "None",
            1: "Minimal outline",
            2: "Partial coverage",
            3: "Moderate coverage",
            4: "Near-complete",
            5: "Fully detailed",
        },
        "version_logs": {
            0: "None",
            1: "Minimal logs",
            2: "Occasional logs",
            3: "Regular logs",
            4: "Near-complete",
            5: "Full version history",
        },
        "collection_protocol": {
            0: "None",
            1: "Vague",
            2: "Partial",
            3: "General methods",
            4: "Well-defined",
            5: "Fully transparent",
        },
        "definition_updates": {
            0: "None",
            1: "Rarely updated",
            2: "Occasional updates",
            3: "Regular but basic",
            4: "Frequent",
            5: "Real-time, documented",
        },
    },
    "dim4": {
        "data_freshness": {
            0: ">5 years old",
            1: "2-5 years old",
            2: "1-2 years old",
            3: "6-12 months old",
            4: "1-6 months old",
            5: "Real-time/current",
        },
        "scheduled_refresh": {
            0: "Never",
            1: "Irregular",
            2: "Annual",
            3: "Quarterly",
            4: "Monthly",
            5: "Daily/real-time",
        },
        "retention_clarity": {
            0: "None",
            1: "Minimal",
            2: "Basic guidelines",
            3: "Moderate clarity",
            4: "High clarity",
            5: "Fully documented",
        },
    },
    "dim5": {
        "regulation_coverage": {
            0: "None",
            1: "Minimal",
            2: "Partial",
            3: "Moderate",
            4: "Comprehensive but dated",
            5: "Fully documented/current",
        },
        "consent_ethics": {
            0: "None",
            1: "Minimal record",
            2: "Partial consent/ethics evidence",
            3: "Moderate logs",
            4: "Substantial",
            5: "Fully documented",
        },
        "geo_restrictions": {
            0: "None",
            1: "Basic mention",
            2: "Partial",
            3: "Moderate",
            4: "Near-complete",
            5: "Fully documented",
        },
        "sensitivity_classification": {
            0: "None",
            1: "Basic flagging",
            2: "Partial",
            3: "Moderate",
            4: "Near-complete",
            5: "Fully classified",
        },
        "audits": {
            0: "None",
            1: "Internal only",
            2: "Basic certification",
            3: "Occasional audit",
            4: "Recent audit",
            5: "Regular external audits",
        },
    },
    "dim6": {
        "license_terms": {
            0: "None",
            1: "Vague",
            2: "Basic",
            3: "Clear",
            4: "Detailed",
            5: "Industry-compliant",
        },
        "ethical_reviews": {
            0: "None",
            1: "Informal approval",
            2: "Partial",
            3: "Moderate",
            4: "Well-documented",
            5: "Certified",
        },
        "redistribution": {
            0: "No policy",
            1: "Unclear",
            2: "Partial",
            3: "Clear",
            4: "Detailed",
            5: "Fully compliant",
        },
        "user_agreements": {
            0: "Non-compliant",
            1: "Minimal adherence",
            2: "Partial",
            3: "Mostly compliant",
            4: "Fully compliant",
            5: "Audited compliance",
        },
    },
}

# ----------------------------------------------------------------------
# FACTOR-SPECIFIC ADEQUACY POLICY
# ----------------------------------------------------------------------
# The minimum adequate rank is derived factor-by-factor from the wording of
# the Appendix-A rubric. It is NOT learned from model/test outcomes.
#
# Healthcare is the primary policy. CIFAR-10 uses the same reference ranks
# for cross-domain comparability; its DQ factors are measured with image-
# specific checks, while documentary dimensions remain controlled evidence.
DQ_RAW_METRIC_BY_FACTOR = {
    "completeness": "missing_fraction",
    "duplication_rate": "duplicate_fraction",
    "value_validity_error_rate": "error_fraction",
    "type_consistency": "type_inconsistency_fraction",
    "label_integrity": "invalid_label_fraction",
    "feature_distribution_consistency": "max_jsd",
    "feature_category_coverage": "mean_category_coverage",
    "structural_constraint_integrity": "structural_violation_fraction",
}

assert abs(sum(WEIGHTS_PSCORE_DEFAULT.values()) - 1.0) < 1e-12
assert sum(len(v) for v in FACTOR_NAMES.values()) == 28
assert len(FACTOR_NAMES["dim1"]) == 4
assert len(FACTOR_NAMES["dim2"]) == 8

# ======================================================================================
# GENERAL UTILITIES
# ======================================================================================

def seed_everything(seed: int):
    seed = int(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        tf.keras.utils.set_random_seed(seed)
    except Exception:
        tf.random.set_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


def sha256_bytes(x: bytes) -> str:
    return hashlib.sha256(x).hexdigest()


def sha256_array(x: np.ndarray) -> str:
    x = np.asarray(x)
    return sha256_bytes(np.ascontiguousarray(x).view(np.uint8).tobytes())


def sha256_weights(weights: List[np.ndarray]) -> str:
    h = hashlib.sha256()
    for w in weights:
        a = np.ascontiguousarray(np.asarray(w))
        h.update(str(a.shape).encode())
        h.update(a.view(np.uint8).tobytes())
    return h.hexdigest()


def _process_rss_mb() -> float:
    try:
        import psutil
        return float(psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2))
    except Exception:
        pass
    try:
        with open("/proc/self/statm", "r", encoding="utf-8") as f:
            pages = int(f.read().split()[1])
        return float(pages * int(os.sysconf("SC_PAGE_SIZE")) / (1024 ** 2))
    except Exception:
        pass
    try:
        import resource
        x = float(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss)
        return x / (1024 ** 2) if sys.platform == "darwin" else x / 1024.0
    except Exception:
        return 0.0


class RAMMonitor:
    def __init__(self, interval_s: float = 0.05):
        self.interval_s = float(interval_s)
        self.start_mb = 0.0
        self.end_mb = 0.0
        self.peak_mb = 0.0
        self._stop = threading.Event()
        self._thread = None

    def _loop(self):
        while not self._stop.wait(self.interval_s):
            self.peak_mb = max(self.peak_mb, _process_rss_mb())

    def start(self):
        self.start_mb = _process_rss_mb()
        self.peak_mb = self.start_mb
        self._stop.clear()
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()
        return self

    def stop(self) -> Dict[str, float]:
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=1.0)
        self.end_mb = _process_rss_mb()
        self.peak_mb = max(self.peak_mb, self.start_mb, self.end_mb)
        return {
            "ram_start_mb": float(self.start_mb),
            "ram_end_mb": float(self.end_mb),
            "ram_peak_mb": float(self.peak_mb),
            "ram_delta_mb": float(max(0.0, self.peak_mb - self.start_mb)),
            "ram_mb": float(self.peak_mb),
        }


def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)
    return Path(path)


def append_csv(row: Dict[str, Any], path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame([row]).to_csv(
        path, mode="a", header=not path.exists(), index=False
    )


def summarize_runs(perf: pd.DataFrame) -> pd.DataFrame:
    if perf.empty:
        return pd.DataFrame()
    metrics = [
        c for c in [
            "accuracy", "precision_macro", "recall_macro", "f1_macro",
            "roc_auc_ovr_macro", "runtime_s", "energy_wh", "communication_mb",
            "optimizer_steps", "participants", "ram_peak_mb", "ram_delta_mb"
        ] if c in perf.columns
    ]
    rows = []
    for scenario, d in perf.groupby("scenario", sort=False):
        row = {"scenario": scenario, "n_runs": len(d)}
        for m in metrics:
            vals = pd.to_numeric(d[m], errors="coerce")
            row[f"{m}_mean"] = float(vals.mean())
            row[f"{m}_sd"] = float(vals.std(ddof=1)) if len(vals) > 1 else 0.0
        rows.append(row)
    return pd.DataFrame(rows)


def package_and_download_results(output_dir: Path, label: str) -> Path:
    output_dir = Path(output_dir)
    manifest = []
    for p in sorted(output_dir.rglob("*")):
        if p.is_file():
            manifest.append({
                "relative_path": str(p.relative_to(output_dir)),
                "bytes": int(p.stat().st_size),
                "sha256": hashlib.sha256(p.read_bytes()).hexdigest(),
            })
    pd.DataFrame(manifest).to_csv(
        output_dir / "RESULTS_FILE_MANIFEST.csv", index=False
    )

    zip_path = output_dir.parent / f"{output_dir.name}_RESULTS.zip"
    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(
        zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6
    ) as zf:
        for p in sorted(output_dir.rglob("*")):
            if p.is_file():
                zf.write(p, arcname=str(p.relative_to(output_dir)))

    print("\n" + "=" * 100)
    print(f"{label} COMPLETE")
    print(f"Results ZIP: {zip_path}")
    print(f"ZIP size: {zip_path.stat().st_size / (1024**2):.2f} MB")
    print("=" * 100)

    try:
        from google.colab import files as colab_files
        print("Starting automatic download to your laptop...")
        colab_files.download(str(zip_path))
    except Exception as exc:
        print("Automatic Colab download unavailable.")
        print(f"ZIP remains at: {zip_path}")
        print(f"Reason: {exc}")

    return zip_path


# ======================================================================================
# CONTROLLED DOCUMENTARY EVIDENCE — v20.0
# ======================================================================================

def rubric_descriptor(dim: str, factor: str, score: float) -> str:
    s = int(np.clip(np.rint(float(score)), 0, 5))
    return str(RUBRIC_DESCRIPTORS[dim][factor][s])


def generate_controlled_documentary_evidence(client_ids, evidence_seed, factor_minima=None, domain="healthcare"):
    rng=np.random.default_rng(int(evidence_seed)); domain=str(domain).lower()
    def set_values(factors,dim,values):
        for f,v in zip(FACTOR_NAMES[dim],values): factors[dim][f]=float(v)
    def make(role,variant):
        factors={d:{} for d in DOCUMENTARY_DIMS}
        if role=="DIRECT_STRONG":
            for d in DOCUMENTARY_DIMS:
                for j,f in enumerate(FACTOR_NAMES[d]): factors[d][f]=5.0 if ((j+variant+len(d))%3==0) else 4.0
        elif role=="REVIEW_RECOVERABLE":
            for d in DOCUMENTARY_DIMS:
                for f in FACTOR_NAMES[d]: factors[d][f]=3.0
            if variant%2: factors["dim3"]["data_dictionary"]=4.0; factors["dim4"]["scheduled_refresh"]=4.0
        elif role=="REVIEW_LIMITED":
            set_values(factors,"dim1",[2,3,2,3]); set_values(factors,"dim3",[5,5,5,5]); set_values(factors,"dim4",[5,5,5]); set_values(factors,"dim5",[2,3,2,3,3]); set_values(factors,"dim6",[2,3,2,3])
        elif role=="EVIDENCE_INVALID":
            for d in DOCUMENTARY_DIMS:
                for f in FACTOR_NAMES[d]: factors[d][f]=4.0
            c=[("dim1","source_reputation"),("dim3","data_dictionary"),("dim4","data_freshness"),("dim5","geo_restrictions"),("dim6","ethical_reviews")]; d,f=c[variant%len(c)]; factors[d][f]=0.0
        elif role=="LOW_HPS_WEAK":
            set_values(factors,"dim1",[2,3,2,3]); set_values(factors,"dim3",[2,3,2,3]); set_values(factors,"dim4",[2,3,3]); set_values(factors,"dim5",[2,3,2,3,3]); set_values(factors,"dim6",[2,3,2,3])
        elif role=="DIMENSION_FLOOR_WEAK":
            for d in DOCUMENTARY_DIMS:
                for f in FACTOR_NAMES[d]: factors[d][f]=3.0
            for f in FACTOR_NAMES["dim4"]: factors["dim4"][f]=2.0
        else: raise ValueError(role)
        return factors
    n=len(client_ids)
    specs=[("DIRECT_STRONG",0),("DIRECT_STRONG",1),("DIRECT_STRONG",2),("DIRECT_STRONG",3),("REVIEW_RECOVERABLE",0),("REVIEW_RECOVERABLE",1),("EVIDENCE_INVALID",0),("REVIEW_LIMITED",0),("LOW_HPS_WEAK",0),("DIMENSION_FLOOR_WEAK",0)] if n==10 else [(r,i//6) for i,r in enumerate((["DIRECT_STRONG","REVIEW_RECOVERABLE","EVIDENCE_INVALID","REVIEW_LIMITED","LOW_HPS_WEAK","DIMENSION_FLOOR_WEAK"]*((n+5)//6))[:n])]
    bundles=[{"bundle_id":f"E{i+1:02d}","profile":r,"variant":v,"factors":make(r,v)} for i,(r,v) in enumerate(specs)]
    assignment=rng.permutation(n); evidence={}; rows=[]
    for pos,cid in enumerate(client_ids):
        b=bundles[int(assignment[pos])]; evidence[cid]={d:dict(b["factors"][d]) for d in DOCUMENTARY_DIMS}
        for d in DOCUMENTARY_DIMS:
            for f,score in b["factors"][d].items(): rows.append({"client":str(cid),"bundle_id":b["bundle_id"],"evidence_profile":b["profile"],"scenario_role":b["profile"],"profile_variant":b["variant"],"dimension":d,"dimension_name":DIMENSION_NAMES[d],"factor":f,"rubric_score_0_5":float(score),"factor_valid_for_gate":bool(np.isfinite(score) and score>0),"is_critical_review_dimension":d in CRITICAL_REVIEW_DIMS,"human_role":"VERIFY_EVIDENCE_ONLY","admission_decision_by":"SERVER_POLICY","evidence_seed":int(evidence_seed),"domain":domain})
    return evidence,pd.DataFrame(rows)


def dimension_scores_from_factors(factors):
    out={}
    for d,names in FACTOR_NAMES.items():
        vals=[]; valid=True
        for f in names:
            if f not in factors.get(d,{}): valid=False; break
            try: v=float(factors[d][f])
            except Exception: valid=False; break
            if not np.isfinite(v): valid=False; break
            vals.append(v)
        out[d]=float(np.mean(vals)) if valid and vals else np.nan
    return out



def compute_hps(
    dimensions: Dict[str, float],
    weights: Dict[str, float] = WEIGHTS_PSCORE_DEFAULT,
) -> float:
    return float(
        sum(float(weights[d]) * float(dimensions[d]) for d in weights)
    )


def all_zero_score_factors(factors):
    return [f"{d}.{f}" for d,names in FACTOR_NAMES.items() for f in names if f in factors.get(d,{}) and np.isfinite(float(factors[d][f])) and float(factors[d][f])<=0]



def evidence_validity_audit(factors, domain):
    """Compatibility alias for the final 28-factor evidence-validity audit."""
    missing=[]; nonfinite=[]; nonpositive=[]
    for d,names in FACTOR_NAMES.items():
        for f in names:
            key=f"{d}.{f}"
            if f not in factors.get(d,{}): missing.append(key); continue
            try:v=float(factors[d][f])
            except Exception: nonfinite.append(key); continue
            if not np.isfinite(v): nonfinite.append(key)
            elif v<=0: nonpositive.append(key)
    return {"passed":not(missing or nonfinite or nonpositive),"missing":missing,"nonfinite":nonfinite,"nonpositive":nonpositive,"factor_count":28}



def hps_review_audit(factors, domain):
    dims=dimension_scores_from_factors(factors); score=float(sum(CRITICAL_REVIEW_WEIGHTS[d]*float(dims[d]) for d in CRITICAL_REVIEW_DIMS))
    return {"hps_review":score,"hps_review_threshold":float(REVIEW_ACCEPT_THRESHOLD),"hps_review_margin":score-float(REVIEW_ACCEPT_THRESHOLD),"critical_dimensions":CRITICAL_REVIEW_DIMS,"weights":CRITICAL_REVIEW_WEIGHTS}



def dimension_floor_failures(
    dimensions: Dict[str, float],
    floor: float = DIMENSION_MIN_FLOOR,
) -> List[str]:
    """Return dimensions that are missing/non-finite or below the policy floor."""
    return [
        dim
        for dim, value in dimensions.items()
        if (not np.isfinite(float(value))) or float(value) < float(floor)
    ]


def tadp_decision(factors, domain, good_cut=GOOD_CUT, high_cut=HIGH_CUT):
    validity=evidence_validity_audit(factors,domain); dims=dimension_scores_from_factors(factors)
    hps=compute_hps(dims) if all(np.isfinite(float(dims[d])) for d in dims) else np.nan
    review=hps_review_audit(factors,domain) if np.isfinite(hps) else {"hps_review":np.nan,"hps_review_margin":np.nan}
    floors=dimension_floor_failures(dims,DIMENSION_MIN_FLOOR) if np.isfinite(hps) else []
    base={"hps":hps,"hps_review":review["hps_review"],"hps_review_midpoint":float(REVIEW_ACCEPT_THRESHOLD),"hps_review_margin":review["hps_review_margin"],"dimensions":dims,
          "evidence_validity_pass":bool(validity["passed"]),"evidence_missing":";".join(validity["missing"]),"evidence_nonfinite":";".join(validity["nonfinite"]),"evidence_nonpositive":";".join(validity["nonpositive"]),
          "dimension_min_floor":DIMENSION_MIN_FLOOR,"dimension_floor_failures":";".join(floors)}
    if not validity["passed"]: return {**base,"decision_path":"EVIDENCE_VALIDITY_GATE","initial_action":"AUTO_REJECT","final_action":"REJECT","status":"AUTO_REJECTED_EVIDENCE_VALIDITY","reason":"All 28 factors must be present, finite, and strictly >0"}
    if floors: return {**base,"decision_path":"DIMENSION_FLOOR","initial_action":"AUTO_REJECT","final_action":"REJECT","status":"AUTO_REJECTED_DIMENSION_FLOOR","reason":"Dimension floor failed: "+';'.join(floors)}
    if hps<float(good_cut): return {**base,"decision_path":"LOW_HPS","initial_action":"AUTO_REJECT","final_action":"REJECT","status":"AUTO_REJECTED_LOW_HPS","reason":f"HPS {hps:.3f} < {good_cut:.3f}"}
    if hps>=float(high_cut): return {**base,"decision_path":"DIRECT_AUTO_ACCEPT","initial_action":"AUTO_ACCEPT","final_action":"ACCEPT","status":"DIRECT_AUTO_ACCEPTED","reason":f"HPS {hps:.3f} >= {high_cut:.3f}"}
    if review["hps_review"]>=REVIEW_ACCEPT_THRESHOLD: return {**base,"decision_path":"HPS_REVIEW_BAND","initial_action":"AUTOMATED_REVIEW","final_action":"ACCEPT","status":"ACCEPTED_AFTER_AUTOMATED_REVIEW","reason":f"HPS_review {review['hps_review']:.3f} >= {REVIEW_ACCEPT_THRESHOLD:.3f}"}
    return {**base,"decision_path":"HPS_REVIEW_BAND","initial_action":"AUTOMATED_REVIEW","final_action":"REJECT","status":"AUTO_REJECTED_HPS_REVIEW","reason":f"HPS_review {review['hps_review']:.3f} < {REVIEW_ACCEPT_THRESHOLD:.3f}"}


def build_tadp_governance(client_ids, documentary_evidence, dq_scores, run, evidence_seed, domain):
    rows=[]
    for cid in client_ids:
        factors={d:dict(documentary_evidence[cid][d]) for d in DOCUMENTARY_DIMS}; factors["dim2"]={f:float(dq_scores[cid][f]) for f in FACTOR_NAMES["dim2"]}; dec=tadp_decision(factors,domain)
        row={"run":int(run),"domain":str(domain),"evidence_seed":int(evidence_seed),"client":str(cid),"hps":dec["hps"],"hps_review":dec["hps_review"],"hps_review_midpoint":dec["hps_review_midpoint"],"hps_review_margin":dec["hps_review_margin"],
             "critical_review_dimensions":";".join(CRITICAL_REVIEW_DIMS),"critical_review_weights_json":json.dumps(CRITICAL_REVIEW_WEIGHTS,sort_keys=True),"evidence_validity_pass":dec["evidence_validity_pass"],"evidence_missing":dec["evidence_missing"],"evidence_nonfinite":dec["evidence_nonfinite"],"evidence_nonpositive":dec["evidence_nonpositive"],"dimension_min_floor":DIMENSION_MIN_FLOOR,"dimension_floor_failures":dec["dimension_floor_failures"],"decision_path":dec["decision_path"],"initial_action":dec["initial_action"],"final_action":dec["final_action"],"status":dec["status"],"reason":dec["reason"]}
        for d,v in dec["dimensions"].items(): row[f"{d}_score_0_5"]=v
        rows.append(row)
    return pd.DataFrame(rows)


def accepted_tadp_vr(governance_df: pd.DataFrame) -> List[str]:
    return governance_df.loc[
        governance_df["final_action"].eq("ACCEPT"), "client"
    ].astype(str).tolist()


# ======================================================================================
# GREAT EXPECTATIONS# ======================================================================================
# GREAT EXPECTATIONS (GX CORE) — NATIVE VALIDATOR BASELINE
# ======================================================================================
# GX is used here in its native role: validate each client's TRAIN-only data against
# a predefined Expectation Suite and use the suite-level success flag as PASS/FAIL.
# There is NO ranking, NO forced-K selection, and NO TADP governance signal in this
# baseline. Great Expectations reports suite success=True only when all configured
# Expectations pass. The suite deliberately uses common technical checks only:
#   1) schema, 2) datatype consistency, 3) required-value ranges,
#   4) missingness, 5) duplicates / unique IDs, 6) label/domain validity,
#   7) structural integrity.
# Distribution checks are intentionally omitted because non-IID client distributions
# are expected in federated learning and should not by themselves constitute failure.
GX_CORE_VERSION = "1.23.0"
GX_VALUE_MOSTLY = 0.99
GX_NONNULL_REFERENCE_MIN = 0.80
GX_NONNULL_TOLERANCE = 0.15


def ensure_great_expectations():
    """Import pinned GX Core; install once in Colab if unavailable."""
    try:
        import great_expectations as gx
        return gx
    except ImportError:
        import subprocess
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q",
            f"great_expectations=={GX_CORE_VERSION}",
        ])
        import great_expectations as gx
        return gx


def _gx_meta(category: str, name: str) -> Dict[str, Any]:
    return {
        "dq_category": str(category),
        "check_name": str(name),
        "baseline": "GX_NATIVE_VALIDATOR",
    }


def _gx_add(suite, specs: List[Dict[str, Any]], expectation, category: str, name: str):
    suite.add_expectation(expectation)
    specs.append({"category": str(category), "name": str(name)})


def _aggregate_nonnull_reference(
    client_frames: Dict[str, pd.DataFrame],
    columns: List[str],
) -> Dict[str, float]:
    total_rows = float(sum(len(df) for df in client_frames.values()))
    out = {}
    for c in columns:
        nonnull = sum(int(df[c].notna().sum()) for df in client_frames.values() if c in df.columns)
        out[c] = float(nonnull / max(1.0, total_rows))
    return out


def build_gx_healthcare_reference(client_frames: Dict[str, pd.DataFrame], preprocessor: Any) -> Dict[str, Any]:
    first = next(iter(client_frames.values()))

    # GX datatype validation is intentionally independent of TADP's model
    # preprocessing type inference.  The preprocessor labels a feature numeric
    # when >=95% of pooled TRAIN non-missing values are parseable; reusing that
    # inferred list inside GX and then requiring 99% parseability per client
    # creates an artificial contradiction.  GX therefore validates datatype
    # consistency only for fields whose numeric meaning is explicit in the
    # Diabetes data schema.
    known_numeric_fields = [
        "time_in_hospital",
        "num_lab_procedures",
        "num_procedures",
        "num_medications",
        "number_outpatient",
        "number_emergency",
        "number_inpatient",
        "number_diagnoses",
    ]
    gx_numeric_cols = [c for c in known_numeric_fields if c in first.columns]

    return {
        "original_columns": list(first.columns),
        "numeric_cols": gx_numeric_cols,
        "feature_cols": list(preprocessor.feature_cols),
        "nonnull_reference": _aggregate_nonnull_reference(
            client_frames, list(preprocessor.feature_cols)
        ),
    }


def build_gx_healthcare_validation_frame(df: pd.DataFrame, preprocessor: Any) -> pd.DataFrame:
    """GX validation view; raw TRAIN records remain the source of all checks."""
    out = df.copy()
    for c in preprocessor.numeric_cols:
        raw = df[c]
        parsed = pd.to_numeric(raw, errors="coerce")
        type_ok = raw.isna() | parsed.notna()
        out[c] = parsed.astype(float)
        out[f"__gx_type_ok__{c}"] = type_ok.astype(np.int8)
    return out


def build_gx_healthcare_suite(gx, reference: Dict[str, Any]):
    """Simple native GX technical-validation suite for the Diabetes TRAIN shards."""
    gxe = gx.expectations
    suite = gx.ExpectationSuite(name="tadp_healthcare_gx_native_suite")
    specs = []
    cols = reference["original_columns"]

    # 1) Schema.
    _gx_add(
        suite, specs,
        gxe.ExpectTableColumnsToMatchSet(
            column_set=cols,
            exact_match=False,
            severity="critical",
            meta=_gx_meta("Schema", "required_column_set"),
        ),
        "Schema", "required_column_set",
    )

    # 2) Datatype consistency for columns inferred as numeric from TRAIN only.
    for c in reference["numeric_cols"]:
        diag = f"__gx_type_ok__{c}"
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeInSet(
                column=diag,
                value_set=[1],
                mostly=GX_VALUE_MOSTLY,
                severity="critical",
                meta=_gx_meta("Datatype Consistency", f"{c}_numeric_parseability"),
            ),
            "Datatype Consistency", f"{c}_numeric_parseability",
        )

    # 3) Required-value ranges for known count/duration fields.
    nonnegative_fields = [
        "num_lab_procedures", "num_procedures", "num_medications",
        "number_outpatient", "number_emergency", "number_inpatient",
        "number_diagnoses",
    ]
    for c in nonnegative_fields:
        if c in cols:
            _gx_add(
                suite, specs,
                gxe.ExpectColumnValuesToBeBetween(
                    column=c, min_value=0.0, mostly=GX_VALUE_MOSTLY,
                    severity="critical",
                    meta=_gx_meta("Required Value Ranges", f"{c}_nonnegative"),
                ),
                "Required Value Ranges", f"{c}_nonnegative",
            )
    if "time_in_hospital" in cols:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeBetween(
                column="time_in_hospital", min_value=1.0, max_value=14.0,
                mostly=GX_VALUE_MOSTLY,
                severity="critical",
                meta=_gx_meta("Required Value Ranges", "time_in_hospital_range"),
            ),
            "Required Value Ranges", "time_in_hospital_range",
        )

    # 4) Missingness. Only columns that are substantially populated in the frozen
    # TRAIN reference are treated as required-enough for a missingness expectation.
    for c, global_nonnull in reference["nonnull_reference"].items():
        if c not in cols or float(global_nonnull) < GX_NONNULL_REFERENCE_MIN:
            continue
        minimum = max(0.70, float(global_nonnull) - GX_NONNULL_TOLERANCE)
        _gx_add(
            suite, specs,
            gxe.ExpectColumnProportionOfNonNullValuesToBeBetween(
                column=c, min_value=float(minimum), max_value=1.0,
                severity="warning",
                meta=_gx_meta("Missingness", f"{c}_nonnull"),
            ),
            "Missingness", f"{c}_nonnull",
        )

    # 5) Duplicates / unique identifiers.
    if "encounter_id" in cols:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeUnique(
                column="encounter_id",
                severity="critical",
                meta=_gx_meta("Duplicates / Unique IDs", "encounter_id_unique"),
            ),
            "Duplicates / Unique IDs", "encounter_id_unique",
        )

    # 6) Labels / domain validity.
    if "_target" in cols:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeInSet(
                column="_target", value_set=[0, 1, 2],
                severity="critical",
                meta=_gx_meta("Labels / Domain Validity", "target_domain"),
            ),
            "Labels / Domain Validity", "target_domain",
        )

    # 7) Structural integrity.
    if "_row_id" in cols:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeUnique(
                column="_row_id",
                severity="critical",
                meta=_gx_meta("Structural Integrity", "row_id_unique"),
            ),
            "Structural Integrity", "row_id_unique",
        )
    _gx_add(
        suite, specs,
        gxe.ExpectTableRowCountToBeBetween(
            min_value=100,
            severity="warning",
            meta=_gx_meta("Structural Integrity", "minimum_client_rows"),
        ),
        "Structural Integrity", "minimum_client_rows",
    )
    return suite, specs


def build_gx_cifar_metadata(X: np.ndarray, y: np.ndarray) -> pd.DataFrame:
    X = np.asarray(X)
    y = np.asarray(y).reshape(-1)
    if X.ndim != 4:
        raise RuntimeError(f"Expected CIFAR tensor [N,H,W,C], got shape={X.shape}")
    finite = np.isfinite(X.astype(np.float32)).reshape(len(X), -1).all(axis=1)
    flat = X.reshape(len(X), -1)
    return pd.DataFrame({
        "sample_id": np.arange(len(X), dtype=np.int64),
        "label": y.astype(np.int32),
        "height": np.full(len(X), X.shape[1], dtype=np.int32),
        "width": np.full(len(X), X.shape[2], dtype=np.int32),
        "channels": np.full(len(X), X.shape[3], dtype=np.int32),
        "dtype_ok": np.full(len(X), int(X.dtype == np.uint8), dtype=np.int8),
        "finite": finite.astype(np.int8),
        "pixel_min": flat.min(axis=1).astype(float),
        "pixel_max": flat.max(axis=1).astype(float),
    })


def build_gx_cifar_reference(client_raw: Dict[str, Tuple[np.ndarray, np.ndarray]]) -> Dict[str, Any]:
    # Native validator uses fixed technical constraints; no distribution reference needed.
    return {}


def build_gx_cifar_suite(gx, reference: Dict[str, Any]):
    """Simple native GX technical-validation suite for CIFAR-10 client metadata."""
    gxe = gx.expectations
    suite = gx.ExpectationSuite(name="tadp_cifar10_gx_native_suite")
    specs = []
    columns = [
        "sample_id", "label", "height", "width", "channels",
        "dtype_ok", "finite", "pixel_min", "pixel_max",
    ]

    _gx_add(
        suite, specs,
        gxe.ExpectTableColumnsToMatchSet(
            column_set=columns, exact_match=True,
            meta=_gx_meta("Schema", "image_metadata_schema"),
        ),
        "Schema", "image_metadata_schema",
    )
    for c in columns:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToNotBeNull(
                column=c,
                severity="warning",
                meta=_gx_meta("Missingness", f"{c}_not_null"),
            ),
            "Missingness", f"{c}_not_null",
        )
    for c, value in [("height", 32), ("width", 32), ("channels", 3), ("dtype_ok", 1), ("finite", 1)]:
        category = "Datatype Consistency" if c in {"dtype_ok", "finite"} else "Structural Integrity"
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeInSet(
                column=c, value_set=[value],
                meta=_gx_meta(category, f"{c}_constraint"),
            ),
            category, f"{c}_constraint",
        )
    for c in ["pixel_min", "pixel_max"]:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeBetween(
                column=c, min_value=0.0, max_value=255.0,
                meta=_gx_meta("Required Value Ranges", f"{c}_valid_range"),
            ),
            "Required Value Ranges", f"{c}_valid_range",
        )
    _gx_add(
        suite, specs,
        gxe.ExpectColumnValuesToBeInSet(
            column="label", value_set=list(range(10)),
            meta=_gx_meta("Labels / Domain Validity", "label_domain"),
        ),
        "Labels / Domain Validity", "label_domain",
    )
    _gx_add(
        suite, specs,
        gxe.ExpectColumnValuesToBeUnique(
            column="sample_id",
            meta=_gx_meta("Duplicates / Unique IDs", "sample_id_unique"),
        ),
        "Duplicates / Unique IDs", "sample_id_unique",
    )
    _gx_add(
        suite, specs,
        gxe.ExpectTableRowCountToBeBetween(
            min_value=100,
            severity="warning",
            meta=_gx_meta("Structural Integrity", "minimum_client_images"),
        ),
        "Structural Integrity", "minimum_client_images",
    )
    return suite, specs


def ge_governance(
    client_data: Dict[str, Any],
    client_ids: List[str],
    accept_count: Optional[int] = None,  # ignored; retained only for call compatibility
    domain: str = "healthcare",
    preprocessor: Optional[Any] = None,
    dq_scores: Optional[Dict[str, Dict[str, float]]] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Run native GX suite validation and PASS only clients whose entire suite succeeds."""
    domain = str(domain).lower()
    gx = ensure_great_expectations()
    context = gx.get_context(mode="ephemeral")
    datasource = context.data_sources.add_pandas(name=f"tadp_gx_native_{domain}_datasource")
    asset = datasource.add_dataframe_asset(name=f"tadp_gx_native_{domain}_asset")
    batch_definition = asset.add_batch_definition_whole_dataframe(name="whole_client_train_partition")

    if domain == "healthcare":
        if preprocessor is None:
            raise ValueError("Healthcare GX native baseline requires TRAIN-only preprocessor metadata.")
        reference = build_gx_healthcare_reference(client_data, preprocessor)
        suite, specs = build_gx_healthcare_suite(gx, reference)
        make_frame = lambda cid: build_gx_healthcare_validation_frame(client_data[cid], preprocessor)
    elif domain == "cifar10":
        reference = build_gx_cifar_reference(client_data)
        suite, specs = build_gx_cifar_suite(gx, reference)
        def make_frame(cid):
            X, y = client_data[cid]
            return build_gx_cifar_metadata(X, y)
    else:
        raise ValueError(f"Unsupported GX domain: {domain!r}")

    summary_rows, detail_rows = [], []
    for cid in client_ids:
        frame = make_frame(cid)
        batch = batch_definition.get_batch(batch_parameters={"dataframe": frame})
        validation = batch.validate(suite)
        results = list(validation.results)
        if len(results) != len(specs):
            raise RuntimeError(
                f"GX result/spec mismatch for client {cid}: {len(results)} vs {len(specs)}"
            )

        passed = 0
        critical_failures = 0
        warning_failures = 0
        info_failures = 0
        category_totals, category_passed = {}, {}
        observed_names = []

        for result in results:
            success = bool(result.success)
            passed += int(success)
            cfg = result.expectation_config
            meta = getattr(cfg, "meta", None) or {}
            name = str(meta.get("check_name", getattr(cfg, "type", "GX_EXPECTATION")))
            cat = str(meta.get("dq_category", "Technical Validation"))

            sev_obj = getattr(cfg, "severity", None)
            sev = getattr(sev_obj, "value", sev_obj)
            sev = str(sev if sev is not None else "critical").lower()
            if "." in sev:
                sev = sev.split(".")[-1]
            if sev not in {"critical", "warning", "info"}:
                sev = "critical"

            if not success:
                if sev == "critical":
                    critical_failures += 1
                elif sev == "warning":
                    warning_failures += 1
                else:
                    info_failures += 1

            observed_names.append(name)
            category_totals[cat] = category_totals.get(cat, 0) + 1
            category_passed[cat] = category_passed.get(cat, 0) + int(success)

            detail_rows.append({
                "client": str(cid),
                "domain": domain,
                "gx_version": GX_CORE_VERSION,
                "expectation_name": name,
                "dq_category": cat,
                "severity": sev,
                "success": success,
            })

        expected_names = sorted(str(x["name"]) for x in specs)
        if sorted(observed_names) != expected_names:
            raise RuntimeError(
                f"GX expectation identity mismatch for client {cid}: "
                f"expected={expected_names}, observed={sorted(observed_names)}"
            )

        total = len(specs)
        stats = getattr(validation, "statistics", {}) or {}
        suite_success = bool(validation.success)

        # GX-native severity-aware operational gate.
        # GX itself exposes the maximum failed severity for a Validation Result.
        # We use that native result rather than reconstructing the gate from a
        # custom ranking.  A failed Expectation execution is also treated by GX
        # as CRITICAL.
        max_failed_severity_obj = validation.get_max_severity_failure()
        if max_failed_severity_obj is None:
            max_failed_severity = "none"
        else:
            max_failed_severity = str(
                getattr(max_failed_severity_obj, "value", max_failed_severity_obj)
            ).lower()
            if "." in max_failed_severity:
                max_failed_severity = max_failed_severity.split(".")[-1]

        operational_valid = (max_failed_severity != "critical")

        row = {
            "client": str(cid),
            "gx_version": GX_CORE_VERSION,
            "gx_native_suite_success": suite_success,
            "gx_operational_valid": bool(operational_valid),
            "gx_critical_failures": int(critical_failures),
            "gx_warning_failures": int(warning_failures),
            "gx_info_failures": int(info_failures),
            "gx_max_failed_severity": max_failed_severity,
            "ge_expectations_passed": int(stats.get("successful_expectations", passed)),
            "ge_expectations_total": int(stats.get("evaluated_expectations", total)),
            "ge_pass_rate": float(
                stats.get("success_percent", 100.0 * passed / max(1, total))
            ) / 100.0,
            "ge_native_validation_class": (
                "GX_SUITE_PASS"
                if suite_success
                else (
                    "GX_WARNING_ONLY"
                    if operational_valid
                    else "GX_CRITICAL_FAILURE"
                )
            ),
            "ge_final_action": "ACCEPT" if operational_valid else "REJECT",
            "ge_policy": (
                "GX Core severity-aware validation; ACCEPT requires zero critical "
                "Expectation failures; warning/info failures are reported but do not "
                "exclude; no ranking and no forced-K selection"
            ),
        }

        for cat in sorted(category_totals):
            safe = re.sub(r"[^a-z0-9]+", "_", cat.lower()).strip("_")
            row[f"ge_{safe}_passed"] = int(category_passed.get(cat, 0))
            row[f"ge_{safe}_total"] = int(category_totals[cat])

        summary_rows.append(row)

    return pd.DataFrame(summary_rows), pd.DataFrame(detail_rows)


def score_lower_is_better(value: float, cuts: List[float]) -> float:
    """
    cuts = [best_upper, score4_upper, score3_upper, score2_upper, score1_upper]
    value <= cuts[0] => 5; ... value <= cuts[4] => 1; else 0.
    """
    v = float(value)
    for score, upper in zip([5, 4, 3, 2, 1], cuts):
        if v <= float(upper):
            return float(score)
    return 0.0


def score_higher_is_better(value: float, cuts: List[float]) -> float:
    """
    cuts = [score5_lower, score4_lower, score3_lower, score2_lower, score1_lower]
    """
    v = float(value)
    for score, lower in zip([5, 4, 3, 2, 1], cuts):
        if v >= float(lower):
            return float(score)
    return 0.0


def js_divergence(p, q, eps=1e-12) -> float:
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    p = p / max(eps, p.sum())
    q = q / max(eps, q.sum())
    m = 0.5 * (p + q)
    kl_pm = np.sum(np.where(p > 0, p * np.log((p + eps) / (m + eps)), 0.0))
    kl_qm = np.sum(np.where(q > 0, q * np.log((q + eps) / (m + eps)), 0.0))
    return float(0.5 * (kl_pm + kl_qm))


# ======================================================================================
# MODEL / METRICS / FL TRAINING
# ======================================================================================

def class_weight_dict(y: np.ndarray) -> Dict[int, float]:
    y = np.asarray(y, dtype=np.int32)
    classes = np.unique(y)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    return {int(c): float(w) for c, w in zip(classes, weights)}


def model_parameter_bytes(model: keras.Model) -> int:
    return int(sum(np.asarray(w).nbytes for w in model.get_weights()))


def evaluate_model(
    model: keras.Model, X: np.ndarray, y: np.ndarray, n_classes: int
) -> Dict[str, float]:
    p = model.predict(X, batch_size=512, verbose=0)
    pred = np.argmax(p, axis=1)
    out = {
        "accuracy": float(accuracy_score(y, pred)),
        "precision_macro": float(
            precision_score(y, pred, average="macro", zero_division=0)
        ),
        "recall_macro": float(
            recall_score(y, pred, average="macro", zero_division=0)
        ),
        "f1_macro": float(
            f1_score(y, pred, average="macro", zero_division=0)
        ),
    }
    try:
        out["roc_auc_ovr_macro"] = float(
            roc_auc_score(y, p, multi_class="ovr", average="macro")
        )
    except Exception:
        out["roc_auc_ovr_macro"] = float("nan")
    return out



ROUND_PROGRESS_MONITOR_MAX_SAMPLES = 4096
ROUND_PROGRESS_POWER_W = 12.0


def build_train_monitor_subset(
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    max_samples: int = ROUND_PROGRESS_MONITOR_MAX_SAMPLES,
    seed: int = 99117,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Build a deterministic lightweight monitoring subset from TRAIN client arrays only.

    This subset is used solely for console progress after FL rounds. It never affects
    preprocessing, governance, client selection, optimizer budgets, checkpoint
    decisions, or final reporting. The held-out TEST set remains final-evaluation-only.
    """
    rng = np.random.default_rng(int(seed))
    client_ids = sorted(client_arrays.keys())
    sizes = {cid: int(len(client_arrays[cid][1])) for cid in client_ids}
    total = int(sum(sizes.values()))
    target = int(min(max_samples, total))

    # Proportional allocation across clients, then distribute rounding remainder.
    raw = {cid: target * sizes[cid] / max(1, total) for cid in client_ids}
    take = {cid: min(sizes[cid], int(np.floor(raw[cid]))) for cid in client_ids}

    while sum(take.values()) < target:
        candidates = [c for c in client_ids if take[c] < sizes[c]]
        if not candidates:
            break
        cid = max(candidates, key=lambda c: (raw[c] - take[c], sizes[c], c))
        take[cid] += 1

    xs, ys = [], []
    for cid in client_ids:
        n_take = int(take[cid])
        if n_take <= 0:
            continue
        Xc, yc = client_arrays[cid]
        if n_take >= len(yc):
            idx = np.arange(len(yc))
        else:
            idx = rng.choice(len(yc), size=n_take, replace=False)
        xs.append(np.asarray(Xc[idx]))
        ys.append(np.asarray(yc[idx], dtype=np.int32))

    Xmon = np.concatenate(xs, axis=0)
    ymon = np.concatenate(ys, axis=0)

    # Deterministic shuffle so monitoring batches do not follow client order.
    order = rng.permutation(len(ymon))
    return Xmon[order], ymon[order]


def _fmt_metric(x: float) -> str:
    return "nan" if not np.isfinite(float(x)) else f"{float(x):.4f}"


def print_round_progress(
    progress_context: Optional[Dict[str, Any]],
    round_idx: int,
    round_total: int,
    selected_ids: List[str],
    round_steps: int,
    cumulative_steps: int,
    monitor_metrics: Dict[str, float],
    cumulative_runtime_s: float,
    cumulative_communication_mb: float,
    ram_start_mb: float,
    ram_peak_mb: float,
    extra: str = "",
) -> None:
    """Compact, human-readable progress block after each federated round."""
    ctx = progress_context or {}
    scenario = str(ctx.get("scenario", "Federated scenario"))
    run_idx = int(ctx.get("run_idx", 0))
    run_total = int(ctx.get("run_total", 0))
    scenario_idx = int(ctx.get("scenario_idx", 0))
    scenario_total = int(ctx.get("scenario_total", 0))
    overall_idx = int(ctx.get("overall_idx", 0))
    overall_total = int(ctx.get("overall_total", 0))

    energy_wh = float(
        ROUND_PROGRESS_POWER_W * float(cumulative_runtime_s) / 3600.0
    )
    ram_delta = max(0.0, float(ram_peak_mb) - float(ram_start_mb))

    print("\n" + "-" * 112)
    print(
        f"PROGRESS | overall configuration {overall_idx}/{overall_total} | "
        f"run {run_idx}/{run_total} | scenario {scenario_idx}/{scenario_total}"
    )
    print(f"SCENARIO | {scenario}")
    print(
        f"ROUND    | {round_idx}/{round_total} | "
        f"selected={len(selected_ids)} [{','.join(map(str, selected_ids))}] | "
        f"steps={round_steps} | cumulative_steps={cumulative_steps}"
    )
    if extra:
        print(f"DETAIL   | {extra}")
    print(
        "TRAIN-MONITOR (diagnostic only; TEST untouched) | "
        f"Accuracy={_fmt_metric(monitor_metrics.get('accuracy', np.nan))} | "
        f"F1={_fmt_metric(monitor_metrics.get('f1_macro', np.nan))} | "
        f"AUC={_fmt_metric(monitor_metrics.get('roc_auc_ovr_macro', np.nan))} | "
        f"Precision={_fmt_metric(monitor_metrics.get('precision_macro', np.nan))} | "
        f"Recall={_fmt_metric(monitor_metrics.get('recall_macro', np.nan))}"
    )
    print(
        f"CUMULATIVE OPERATIONAL | runtime={cumulative_runtime_s:.2f}s | "
        f"energy≈{energy_wh:.5f}Wh | communication={cumulative_communication_mb:.3f}MB | "
        f"RAM peak={ram_peak_mb:.1f}MB | RAM Δ={ram_delta:.1f}MB"
    )
    print("-" * 112)


def train_exact_steps(
    model: keras.Model,
    X: np.ndarray,
    y: np.ndarray,
    steps: int,
    batch_size: int,
    seed: int,
    class_weights: Optional[Dict[int, float]] = None,
    prox_reference: Optional[List[np.ndarray]] = None,
    prox_mu: float = 0.0,
):
    """
    Exact mini-batch update count. Used for step-parity audits.
    """
    steps = int(max(1, steps))
    rng = np.random.default_rng(int(seed))
    n = len(y)
    if n == 0:
        raise RuntimeError("Cannot train on an empty dataset.")

    loss_fn = keras.losses.SparseCategoricalCrossentropy(
        reduction=keras.losses.Reduction.NONE
    )

    order = rng.permutation(n)
    cursor = 0

    prox_tensors = None
    if prox_reference is not None and prox_mu > 0:
        prox_tensors = [tf.convert_to_tensor(w) for w in prox_reference]

    for _ in range(steps):
        if cursor + batch_size > n:
            order = rng.permutation(n)
            cursor = 0

        idx = order[cursor:cursor + batch_size]
        cursor += batch_size

        xb = tf.convert_to_tensor(np.asarray(X[idx]), dtype=tf.float32)
        yb_np = np.asarray(y[idx], dtype=np.int32)
        yb = tf.convert_to_tensor(yb_np, dtype=tf.int32)

        with tf.GradientTape() as tape:
            probs = model(xb, training=True)
            per_loss = loss_fn(yb, probs)

            if class_weights:
                sw = np.array(
                    [class_weights.get(int(v), 1.0) for v in yb_np],
                    dtype=np.float32,
                )
                sw_t = tf.convert_to_tensor(sw)
                data_loss = tf.reduce_sum(per_loss * sw_t) / tf.reduce_sum(sw_t)
            else:
                data_loss = tf.reduce_mean(per_loss)

            loss = data_loss

            if prox_tensors is not None:
                prox = tf.constant(0.0, dtype=tf.float32)
                for var, ref in zip(model.trainable_variables, prox_tensors):
                    prox += tf.reduce_sum(tf.square(var - tf.cast(ref, var.dtype)))
                loss = loss + 0.5 * float(prox_mu) * prox

        grads = tape.gradient(loss, model.trainable_variables)
        model.optimizer.apply_gradients(zip(grads, model.trainable_variables))


def natural_steps(n_records: int, batch_size: int, local_epochs: int = 1) -> int:
    return int(max(1, math.ceil(int(n_records) / int(batch_size)) * int(local_epochs)))


def allocate_exact_step_budget(
    selected: List[str],
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    target_total: int,
    batch_size: int,
    local_epochs: int = 1,
) -> Dict[str, int]:
    natural = {
        cid: natural_steps(len(client_arrays[cid][1]), batch_size, local_epochs)
        for cid in selected
    }
    total_nat = max(1, sum(natural.values()))
    raw = {cid: target_total * natural[cid] / total_nat for cid in selected}
    alloc = {cid: max(1, int(math.floor(raw[cid]))) for cid in selected}

    # Adjust to exact target.
    while sum(alloc.values()) < target_total:
        cid = max(selected, key=lambda c: raw[c] - alloc[c])
        alloc[cid] += 1
    while sum(alloc.values()) > target_total:
        candidates = [c for c in selected if alloc[c] > 1]
        if not candidates:
            break
        cid = min(candidates, key=lambda c: raw[c] - alloc[c])
        alloc[cid] -= 1

    if sum(alloc.values()) != int(target_total):
        raise RuntimeError("Exact step-budget allocation failed.")
    return alloc


def aggregate_weights(
    local_weights: List[List[np.ndarray]],
    sample_sizes: List[int],
    equal_weight: bool = False,
) -> List[np.ndarray]:
    if not local_weights:
        raise RuntimeError("No local weights to aggregate.")
    if equal_weight:
        alpha = np.ones(len(local_weights), dtype=float) / len(local_weights)
    else:
        sizes = np.asarray(sample_sizes, dtype=float)
        alpha = sizes / sizes.sum()

    out = []
    for layer_idx in range(len(local_weights[0])):
        x = sum(alpha[j] * np.asarray(local_weights[j][layer_idx])
                for j in range(len(local_weights)))
        out.append(np.asarray(x))
    return out


def federated_train(
    build_model_fn,
    initial_weights: List[np.ndarray],
    selected_per_round: List[List[str]],
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    X_test: np.ndarray,
    y_test: np.ndarray,
    n_classes: int,
    batch_size: int,
    local_epochs: int,
    class_weights: Dict[int, float],
    run_seed: int,
    exact_step_maps: Optional[List[Dict[str, int]]] = None,
    equal_weight: bool = False,
    fedprox_mu: float = 0.0,
    train_monitor: Optional[Tuple[np.ndarray, np.ndarray]] = None,
    progress_context: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    monitor = RAMMonitor().start()
    start = time.perf_counter()

    global_model = build_model_fn()
    global_model.set_weights([np.array(w, copy=True) for w in initial_weights])

    round_rows = []
    total_steps = 0
    total_selected = 0
    monitor_eval_s = 0.0
    cumulative_comm_raw_b = 0
    param_b = model_parameter_bytes(global_model)

    for r, selected in enumerate(selected_per_round, start=1):
        global_weights = [np.array(w, copy=True) for w in global_model.get_weights()]
        local_weights = []
        local_sizes = []
        round_steps = 0

        if exact_step_maps is None:
            step_map = {
                cid: natural_steps(
                    len(client_arrays[cid][1]), batch_size, local_epochs
                )
                for cid in selected
            }
        else:
            step_map = exact_step_maps[r - 1]

        for j, cid in enumerate(selected):
            Xc, yc = client_arrays[cid]
            local_model = build_model_fn()
            local_model.set_weights(global_weights)
            train_exact_steps(
                local_model, Xc, yc,
                steps=int(step_map[cid]),
                batch_size=batch_size,
                seed=int(run_seed + 1000 * r + 17 * j),
                class_weights=class_weights,
                prox_reference=global_weights if fedprox_mu > 0 else None,
                prox_mu=float(fedprox_mu),
            )
            local_weights.append(
                [np.array(w, copy=True) for w in local_model.get_weights()]
            )
            local_sizes.append(len(yc))
            round_steps += int(step_map[cid])

            del local_model
            tf.keras.backend.clear_session()

        agg = aggregate_weights(
            local_weights, local_sizes, equal_weight=equal_weight
        )
        global_model = build_model_fn()
        global_model.set_weights(agg)

        total_steps += round_steps
        total_selected += len(selected)

        # Cumulative model traffic through this round. Same accounting as the
        # final experiment metric: logical model download + upload payload only.
        cumulative_comm_raw_b += int(len(selected)) * 2 * int(param_b)
        cumulative_comm_mb = float(
            cumulative_comm_raw_b / 1_000_000
        )

        # Measure training runtime BEFORE this round's diagnostic evaluation.
        # Previous diagnostic-evaluation time is subtracted so progress printing
        # does not inflate the experiment's runtime metric.
        cumulative_runtime_s = float(
            max(0.0, (time.perf_counter() - start) - monitor_eval_s)
        )

        if train_monitor is not None:
            Xmon, ymon = train_monitor
            t_mon = time.perf_counter()
            monitor_metrics = evaluate_model(global_model, Xmon, ymon, n_classes)
            monitor_eval_s += float(time.perf_counter() - t_mon)
        else:
            monitor_metrics = {
                "accuracy": float("nan"),
                "precision_macro": float("nan"),
                "recall_macro": float("nan"),
                "f1_macro": float("nan"),
                "roc_auc_ovr_macro": float("nan"),
            }

        live_peak = float(monitor.peak_mb)
        round_row = {
            "round": r,
            "selected_clients": len(selected),
            "selected_ids": ";".join(selected),
            "optimizer_steps": int(round_steps),
            "cumulative_optimizer_steps": int(total_steps),
            "train_monitor_accuracy": float(monitor_metrics["accuracy"]),
            "train_monitor_precision_macro": float(monitor_metrics["precision_macro"]),
            "train_monitor_recall_macro": float(monitor_metrics["recall_macro"]),
            "train_monitor_f1_macro": float(monitor_metrics["f1_macro"]),
            "train_monitor_auc_ovr_macro": float(monitor_metrics["roc_auc_ovr_macro"]),
            "cumulative_runtime_s": float(cumulative_runtime_s),
            "cumulative_energy_wh_est": float(
                ROUND_PROGRESS_POWER_W * cumulative_runtime_s / 3600.0
            ),
            "cumulative_communication_mb": float(cumulative_comm_mb),
            "ram_peak_mb_live": float(live_peak),
            "ram_delta_mb_live": float(max(0.0, live_peak - monitor.start_mb)),
        }
        round_rows.append(round_row)

        print_round_progress(
            progress_context=progress_context,
            round_idx=r,
            round_total=len(selected_per_round),
            selected_ids=list(selected),
            round_steps=int(round_steps),
            cumulative_steps=int(total_steps),
            monitor_metrics=monitor_metrics,
            cumulative_runtime_s=cumulative_runtime_s,
            cumulative_communication_mb=cumulative_comm_mb,
            ram_start_mb=float(monitor.start_mb),
            ram_peak_mb=live_peak,
        )

    runtime = float(max(0.0, (time.perf_counter() - start) - monitor_eval_s))
    metrics = evaluate_model(global_model, X_test, y_test, n_classes)
    ram = monitor.stop()

    communication_b = int(cumulative_comm_raw_b)

    return {
        "model": global_model,
        "metrics": metrics,
        "runtime_s": float(runtime),
        "total_optimizer_steps": int(total_steps),
        "communication_payload_bytes": int(communication_b),
        "communication_mb": float(communication_b / 1_000_000),
        "communication_mib": float(communication_b / (1024 ** 2)),
        "round_audit": round_rows,
        "participants_mean_per_round": float(
            total_selected / max(1, len(round_rows))
        ),
        **ram,
    }



def select_dq_only_clients(
    dq_scores: Dict[str, Dict[str, float]],
    k: int,
) -> Tuple[List[str], pd.DataFrame]:
    """Select top-K clients by the machine-measured TADP Data Quality dimension only."""
    rows = []
    for cid, scores in dq_scores.items():
        vals = [float(scores[f]) for f in FACTOR_NAMES["dim2"]]
        rows.append({
            "client": str(cid),
            "dq_only_score": float(np.mean(vals)),
            "dq_factor_count": int(len(vals)),
        })
    audit = pd.DataFrame(rows).sort_values(
        ["dq_only_score", "client"], ascending=[False, True]
    ).reset_index(drop=True)
    audit["dq_only_rank"] = np.arange(1, len(audit) + 1)
    selected = audit.head(int(k))["client"].astype(str).tolist()
    audit["dq_only_selected"] = audit["client"].isin(selected)
    return selected, audit


def local_training_loss(model: keras.Model, X: np.ndarray, y: np.ndarray, batch_size: int = 512) -> float:
    """Mean sparse cross-entropy on client TRAIN data only; used by Power-of-Choice."""
    p = model.predict(X, batch_size=batch_size, verbose=0)
    y = np.asarray(y, dtype=np.int32)
    idx = np.arange(len(y))
    probs = np.clip(p[idx, y], 1e-12, 1.0)
    return float(-np.mean(np.log(probs)))


def result_row(
    run: int,
    seed: int,
    scenario: str,
    result: Dict[str, Any],
    initial_hash: str,
    power_w: float = 12.0,
) -> Dict[str, Any]:
    row = {
        "run": int(run),
        "seed": int(seed),
        "scenario": str(scenario),
        **result["metrics"],
        "runtime_s": float(result["runtime_s"]),
        "optimizer_steps": int(result["total_optimizer_steps"]),
        "communication_mb": float(result.get("communication_mb", 0.0)),
        "participants": float(result.get("participants_mean_per_round", 0.0)),
        "ram_start_mb": float(result.get("ram_start_mb", 0.0)),
        "ram_end_mb": float(result.get("ram_end_mb", 0.0)),
        "ram_peak_mb": float(result.get("ram_peak_mb", 0.0)),
        "ram_delta_mb": float(result.get("ram_delta_mb", 0.0)),
        "ram_mb": float(result.get("ram_mb", result.get("ram_peak_mb", 0.0))),
        "initial_weights_sha256": initial_hash,
    }
    row["energy_wh"] = float(power_w * row["runtime_s"] / 3600.0)
    row["energy_kwh"] = float(row["energy_wh"] / 1000.0)
    row["co2_kg"] = float(row["energy_kwh"] * 0.445)
    row["energy_cost_usd"] = float(row["energy_kwh"] * 0.20)
    row["communication_cost_usd"] = float(row["communication_mb"] * 0.005)
    row["total_estimated_cost_usd"] = float(
        row["energy_cost_usd"] + row["communication_cost_usd"]
    )
    return row


# ======================================================================================
# LEAKAGE AUDIT
# ======================================================================================

def write_leakage_audit(
    out_dir: Path,
    train_ids,
    test_ids,
    client_train_ids: Dict[str, np.ndarray],
    extra: Optional[Dict[str, Any]] = None,
):
    train_set = set(map(str, train_ids))
    test_set = set(map(str, test_ids))
    overlap = train_set & test_set

    client_union = set()
    duplicates_across_clients = 0
    for cid, ids in client_train_ids.items():
        s = set(map(str, ids))
        duplicates_across_clients += len(client_union & s)
        client_union |= s

    test_in_clients = len(test_set & client_union)
    missing_train = len(train_set - client_union)
    extra_client_rows = len(client_union - train_set)

    row = {
        "train_test_overlap": len(overlap),
        "test_rows_in_any_client": test_in_clients,
        "train_rows_missing_from_clients": missing_train,
        "client_rows_not_in_global_train": extra_client_rows,
        "duplicate_train_rows_across_clients": duplicates_across_clients,
        "pass": (
            len(overlap) == 0
            and test_in_clients == 0
            and missing_train == 0
            and extra_client_rows == 0
            and duplicates_across_clients == 0
        ),
    }
    if extra:
        row.update(extra)

    pd.DataFrame([row]).to_csv(
        Path(out_dir) / "leakage_audit.csv", index=False
    )

    if not bool(row["pass"]):
        raise RuntimeError(f"FAIL-CLOSED leakage audit failed: {row}")
    return row


# ======================================================================================
# DIABETES 130-US — LEAKAGE-SAFE DATA PREPARATION
# ======================================================================================

TARGET = "readmitted"
ID_COLUMNS = ["encounter_id", "patient_nbr"]


def locate_diabetes_csv() -> str:
    candidates = [
        os.environ.get("DIABETES_CSV", ""),
        "/content/diabetes_130US.csv",
        "./diabetes_130US.csv",
        "/content/drive/MyDrive/diabetes_130US.csv",
    ]
    for p in candidates:
        if p and os.path.exists(p):
            return p

    try:
        from google.colab import files as colab_files
        print("Please upload the Diabetes 130-US CSV file.")
        uploaded = colab_files.upload()
        csvs = [name for name in uploaded if str(name).lower().endswith(".csv")]
        if not csvs:
            raise RuntimeError("No CSV file was uploaded.")
        return str(csvs[0])
    except ImportError:
        pass

    raise FileNotFoundError(
        "Diabetes CSV not found. Set DIABETES_CSV or upload the CSV in Colab."
    )


def load_diabetes(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    df = df.replace("?", np.nan)

    if TARGET not in df.columns:
        raise RuntimeError(f"Missing target column: {TARGET}")

    valid_target = {"NO": 0, ">30": 1, "<30": 2}
    df = df[df[TARGET].isin(valid_target)].copy()
    df["_target"] = df[TARGET].map(valid_target).astype(np.int32)
    df["_row_id"] = np.arange(len(df), dtype=np.int64)

    return df


def global_patient_grouped_split(
    df: pd.DataFrame, seed: int, test_fraction: float = 0.20
):
    """
    Patient-grouped and stratified whenever patient_nbr is available.
    The split happens before any client partitioning or data-dependent preprocessing.
    """
    if "patient_nbr" in df.columns:
        splitter = StratifiedGroupKFold(
            n_splits=5, shuffle=True, random_state=int(seed)
        )
        train_idx, test_idx = next(
            splitter.split(
                np.zeros(len(df)),
                y=df["_target"].to_numpy(),
                groups=df["patient_nbr"].astype(str).to_numpy(),
            )
        )
        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()

        patient_overlap = len(
            set(train_df["patient_nbr"].astype(str))
            & set(test_df["patient_nbr"].astype(str))
        )
        if patient_overlap != 0:
            raise RuntimeError("Patient leakage detected across TRAIN/TEST.")
    else:
        train_df, test_df = train_test_split(
            df, test_size=float(test_fraction),
            stratify=df["_target"], random_state=int(seed)
        )
        patient_overlap = 0

    return train_df.reset_index(drop=True), test_df.reset_index(drop=True), patient_overlap


def dirichlet_partition_dataframe(
    train_df: pd.DataFrame,
    n_clients: int,
    alpha: float,
    seed: int,
    min_client_records: int = 100,
) -> Dict[str, pd.DataFrame]:
    y = train_df["_target"].to_numpy()
    client_ids = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")[:n_clients]

    for attempt in range(100):
        rng = np.random.default_rng(int(seed + attempt))
        buckets = [[] for _ in range(n_clients)]

        for cls in sorted(np.unique(y)):
            idx = np.where(y == cls)[0]
            rng.shuffle(idx)
            props = rng.dirichlet(np.full(n_clients, float(alpha)))
            cuts = (np.cumsum(props)[:-1] * len(idx)).astype(int)
            splits = np.split(idx, cuts)
            for k, s in enumerate(splits):
                buckets[k].extend(s.tolist())

        sizes = [len(b) for b in buckets]
        if min(sizes) >= int(min_client_records):
            out = {}
            for cid, idxs in zip(client_ids, buckets):
                out[cid] = train_df.iloc[np.array(idxs, dtype=int)].copy()
            return out

    raise RuntimeError("Could not obtain a valid Dirichlet client partition.")


@dataclass
class TabularPreprocessor:
    feature_cols: List[str]
    numeric_cols: List[str]
    categorical_cols: List[str]
    mean: Dict[str, float]
    std: Dict[str, float]
    categories: Dict[str, List[str]]
    encoder: Any

    def transform(self, df: pd.DataFrame) -> np.ndarray:
        pieces = []

        if self.numeric_cols:
            x_num = []
            for c in self.numeric_cols:
                s = pd.to_numeric(df[c], errors="coerce").astype(float)
                a = s.fillna(self.mean[c]).to_numpy(dtype=np.float32)
                a = (a - self.mean[c]) / self.std[c]
                x_num.append(a[:, None])
            pieces.append(np.concatenate(x_num, axis=1).astype(np.float32))

        if self.categorical_cols:
            cat = pd.DataFrame({
                c: df[c].astype("string").fillna("__MISSING__").astype(str)
                for c in self.categorical_cols
            })
            x_cat = self.encoder.transform(cat)
            pieces.append(np.asarray(x_cat, dtype=np.float32))

        if not pieces:
            raise RuntimeError("No predictor columns remained.")
        return np.concatenate(pieces, axis=1).astype(np.float32)


def fit_federated_train_only_preprocessor(
    client_frames: Dict[str, pd.DataFrame]
) -> TabularPreprocessor:
    """
    No raw TRAIN pooling is used to ESTIMATE numeric parameters.
    Numeric mean/std comes from aggregated local count/sum/sum-of-squares.
    Categorical vocabulary comes from union of local TRAIN category sets.
    """
    any_df = next(iter(client_frames.values()))
    feature_cols = [
        c for c in any_df.columns
        if c not in {TARGET, "_target", "_row_id", *ID_COLUMNS}
    ]

    # Infer expected type from TRAIN only.
    numeric_cols = []
    categorical_cols = []
    for c in feature_cols:
        total_nonmissing = 0
        numeric_valid = 0
        for df in client_frames.values():
            raw = df[c]
            nm = raw.notna()
            total_nonmissing += int(nm.sum())
            if nm.any():
                numeric_valid += int(
                    pd.to_numeric(raw[nm], errors="coerce").notna().sum()
                )
        ratio = numeric_valid / max(1, total_nonmissing)
        if ratio >= 0.95:
            numeric_cols.append(c)
        else:
            categorical_cols.append(c)

    mean = {}
    std = {}
    for c in numeric_cols:
        count = 0
        sum_ = 0.0
        sumsq = 0.0
        for df in client_frames.values():
            a = pd.to_numeric(df[c], errors="coerce").to_numpy(dtype=float)
            a = a[np.isfinite(a)]
            count += len(a)
            sum_ += float(a.sum())
            sumsq += float(np.square(a).sum())
        mu = sum_ / max(1, count)
        var = max(1e-12, sumsq / max(1, count) - mu * mu)
        mean[c] = float(mu)
        std[c] = float(math.sqrt(var))

    categories = {}
    for c in categorical_cols:
        values = set()
        for df in client_frames.values():
            s = df[c].astype("string").fillna("__MISSING__").astype(str)
            values.update(s.unique().tolist())
        categories[c] = sorted(values)

    encoder = OneHotEncoder(
        categories=[categories[c] for c in categorical_cols],
        handle_unknown="ignore",
        sparse_output=False,
        dtype=np.float32,
    )

    # Fit only metadata-shaped dummy rows; the vocabulary is already frozen from TRAIN.
    if categorical_cols:
        max_len = max(len(categories[c]) for c in categorical_cols)
        dummy = {}
        for c in categorical_cols:
            vals = categories[c]
            dummy[c] = [vals[i % len(vals)] for i in range(max_len)]
        encoder.fit(pd.DataFrame(dummy))

    return TabularPreprocessor(
        feature_cols=feature_cols,
        numeric_cols=numeric_cols,
        categorical_cols=categorical_cols,
        mean=mean,
        std=std,
        categories=categories,
        encoder=encoder,
    )


def build_tabular_reference(
    client_frames: Dict[str, pd.DataFrame],
    preprocessor: TabularPreprocessor,
) -> Dict[str, Any]:
    """
    TRAIN-only DQ reference built from client-local summaries only.

    Numerical reference histograms are constructed by combining local
    min/max summaries and then summing client-local histogram counts.
    Categorical reference support is the union of local TRAIN category sets.
    Raw client records are not concatenated to construct the reference.
    """
    ref = {"num_hist": {}, "cat_values": {}}

    chosen_num = preprocessor.numeric_cols[:12]
    for c in chosen_num:
        local_min, local_max = [], []
        for df in client_frames.values():
            a = pd.to_numeric(df[c], errors="coerce").to_numpy(dtype=float)
            a = a[np.isfinite(a)]
            if len(a):
                local_min.append(float(np.min(a)))
                local_max.append(float(np.max(a)))
        if not local_min:
            continue
        gmin, gmax = float(min(local_min)), float(max(local_max))
        if gmax <= gmin:
            edges = np.array([gmin - 1e-6, gmax + 1e-6], dtype=float)
        else:
            edges = np.linspace(gmin, gmax, 11, dtype=float)
        global_hist = np.zeros(len(edges)-1, dtype=np.float64)
        for df in client_frames.values():
            a = pd.to_numeric(df[c], errors="coerce").to_numpy(dtype=float)
            a = a[np.isfinite(a)]
            if len(a):
                h, _ = np.histogram(a, bins=edges)
                global_hist += h.astype(np.float64)
        ref["num_hist"][c] = {
            "edges": edges.tolist(),
            "hist": global_hist.tolist(),
            "construction": "aggregated_client_local_histograms_only",
        }

    for c in preprocessor.categorical_cols:
        values = set()
        for df in client_frames.values():
            values.update(
                df[c].astype("string").fillna("__MISSING__")
                .astype(str).unique().tolist()
            )
        ref["cat_values"][c] = sorted(values)
    return ref

def tabular_dq_scores(
    df: pd.DataFrame,
    preprocessor: TabularPreprocessor,
    reference: Dict[str, Any],
) -> Tuple[Dict[str, float], Dict[str, float]]:
    """
    Eight machine-measured TRAIN-only DQ factors for the healthcare experiment.
    """
    feature_df = df[preprocessor.feature_cols]

    # 1) Completeness.
    missing_fraction = float(feature_df.isna().mean().mean())
    completeness = score_lower_is_better(
        missing_fraction,
        [0.01, 0.05, 0.10, 0.20, 0.50],
    )

    # 2) Duplication rate.
    duplicate_fraction = float(feature_df.duplicated().mean())
    duplication = score_lower_is_better(
        duplicate_fraction,
        [0.01, 0.02, 0.05, 0.10, 0.20],
    )

    # 3) Value validity / error rate.
    bad = 0
    observed = 0

    for c in preprocessor.numeric_cols:
        raw = df[c]
        nm = raw.notna()
        observed += int(nm.sum())

        if nm.any():
            conv = pd.to_numeric(
                raw[nm], errors="coerce"
            ).to_numpy(dtype=float)
            bad += int(np.sum(~np.isfinite(conv)))

    for c in preprocessor.categorical_cols:
        raw = df[c]
        nm = raw.notna()
        observed += int(nm.sum())

        if nm.any():
            bad += int(
                np.sum(raw[nm].astype(str).str.strip().eq(""))
            )

    error_fraction = float(bad / max(1, observed))
    value_validity_error_rate = score_lower_is_better(
        error_fraction,
        [0.01, 0.02, 0.05, 0.10, 0.15],
    )

    # 4) Type consistency.
    type_bad = 0
    type_obs = 0

    for c in preprocessor.numeric_cols:
        raw = df[c]
        nm = raw.notna()
        type_obs += int(nm.sum())

        if nm.any():
            type_bad += int(
                pd.to_numeric(
                    raw[nm], errors="coerce"
                ).isna().sum()
            )

    type_inconsistency = float(type_bad / max(1, type_obs))
    type_consistency = score_lower_is_better(
        type_inconsistency,
        [0.01, 0.02, 0.05, 0.10, 0.20],
    )

    # 5) Label integrity.
    invalid_label_fraction = float(
        (~df["_target"].isin([0, 1, 2])).mean()
    )
    label_integrity = score_lower_is_better(
        invalid_label_fraction,
        [0.001, 0.01, 0.02, 0.05, 0.10],
    )

    # 6) Feature-distribution consistency — TRAIN-only reference.
    jsds = []

    for c, spec in reference["num_hist"].items():
        a = pd.to_numeric(
            df[c], errors="coerce"
        ).to_numpy(dtype=float)
        a = a[np.isfinite(a)]

        if len(a):
            hist, _ = np.histogram(
                a,
                bins=np.array(spec["edges"], dtype=float),
            )
            jsds.append(
                js_divergence(
                    hist,
                    np.array(spec["hist"], dtype=float),
                )
            )

    max_jsd = float(max(jsds)) if jsds else 0.0
    distribution_consistency = score_lower_is_better(
        max_jsd,
        [0.01, 0.025, 0.05, 0.10, 0.20],
    )

    # 7) Feature/category coverage.
    coverage_vals = []

    for c, ref_vals in reference["cat_values"].items():
        ref_set = set(ref_vals)

        if ref_set:
            client_set = set(
                df[c]
                .astype("string")
                .fillna("__MISSING__")
                .astype(str)
                .unique()
            )
            coverage_vals.append(
                len(client_set & ref_set) / len(ref_set)
            )

    mean_coverage = (
        float(np.mean(coverage_vals))
        if coverage_vals else 1.0
    )
    feature_coverage = score_higher_is_better(
        mean_coverage,
        [0.90, 0.825, 0.75, 0.65, 0.50],
    )

    # 8) Structural / constraint integrity.
    # Uses only TRAIN records. It checks:
    #   - key presence / encounter uniqueness;
    #   - non-negative count-like clinical fields;
    #   - finite numeric values where a numeric value is expected.
    n = len(df)
    record_violation = np.zeros(n, dtype=bool)

    if "encounter_id" not in df.columns:
        record_violation[:] = True
    else:
        encounter = df["encounter_id"]
        record_violation |= encounter.isna().to_numpy()
        record_violation |= encounter.duplicated(keep=False).to_numpy()

    nonnegative_fields = [
        "time_in_hospital",
        "num_lab_procedures",
        "num_procedures",
        "num_medications",
        "number_outpatient",
        "number_emergency",
        "number_inpatient",
        "number_diagnoses",
    ]

    for c in nonnegative_fields:
        if c in df.columns:
            a = pd.to_numeric(
                df[c], errors="coerce"
            ).to_numpy(dtype=float)
            bad_c = (~np.isfinite(a)) | (a < 0)
            record_violation |= bad_c

    structural_violation_fraction = float(
        np.mean(record_violation)
    ) if n else 1.0

    structural_integrity = score_lower_is_better(
        structural_violation_fraction,
        [0.001, 0.01, 0.02, 0.05, 0.10],
    )

    scores = {
        "completeness": completeness,
        "duplication_rate": duplication,
        "value_validity_error_rate": value_validity_error_rate,
        "type_consistency": type_consistency,
        "label_integrity": label_integrity,
        "feature_distribution_consistency": distribution_consistency,
        "feature_category_coverage": feature_coverage,
        "structural_constraint_integrity": structural_integrity,
    }

    raw = {
        "missing_fraction": missing_fraction,
        "duplicate_fraction": duplicate_fraction,
        "error_fraction": error_fraction,
        "type_inconsistency_fraction": type_inconsistency,
        "invalid_label_fraction": invalid_label_fraction,
        "max_jsd": max_jsd,
        "mean_category_coverage": mean_coverage,
        "structural_violation_fraction": structural_violation_fraction,
    }

    return scores, raw

@dataclass
class TabularDQSchema:
    feature_cols: List[str]
    numeric_cols: List[str]
    categorical_cols: List[str]

def infer_train_only_dq_schema(client_frames):
    any_df=next(iter(client_frames.values())); feature=[c for c in any_df.columns if c not in {TARGET,"_target","_row_id",*ID_COLUMNS}]; num=[]; cat=[]
    for c in feature:
        total=valid=0
        for df in client_frames.values():
            raw=df[c]; nm=raw.notna(); total+=int(nm.sum()); valid+=int(pd.to_numeric(raw[nm],errors="coerce").notna().sum()) if nm.any() else 0
        (num if valid/max(1,total)>=0.95 else cat).append(c)
    return TabularDQSchema(feature,num,cat)

def preprocessor_fingerprint(pre):
    payload={"feature_cols":pre.feature_cols,"numeric_cols":pre.numeric_cols,"categorical_cols":pre.categorical_cols,"mean":pre.mean,"std":pre.std,"categories":pre.categories}
    return hashlib.sha256(json.dumps(payload,sort_keys=True,default=str).encode()).hexdigest()

def build_post_selection_attack_view(client_frames,test_df,selected_clients):
    ids=sorted(map(str,selected_clients)); selected={c:client_frames[c] for c in ids}; pre=fit_federated_train_only_preprocessor(selected)
    arrays={c:(pre.transform(selected[c]),selected[c]["_target"].to_numpy(np.int32)) for c in ids}; X_test=pre.transform(test_df); y_test=test_df["_target"].to_numpy(np.int32); cw=class_weight_dict(np.concatenate([arrays[c][1] for c in ids]))
    return {"selected_clients":ids,"preprocessor":pre,"preprocessor_sha256":preprocessor_fingerprint(pre),"client_arrays":arrays,"X_test":X_test,"y_test":y_test,"class_weights":cw,"input_dim":int(X_test.shape[1])}

def prepare_diabetes_no_leakage(csv_path, split_seed, partition_seed, n_clients=10, alpha=1.0):
    raw=load_diabetes(csv_path); train_df,test_df,patient_overlap=global_patient_grouped_split(raw,split_seed); clients=dirichlet_partition_dataframe(train_df,n_clients=n_clients,alpha=alpha,seed=partition_seed); ids=list(clients)
    dq_schema=infer_train_only_dq_schema(clients); reference=build_tabular_reference(clients,dq_schema); dq_scores={}; rows=[]
    for cid,df in clients.items():
        sc,rawm=tabular_dq_scores(df,dq_schema,reference); dq_scores[cid]=sc; rows.append({"client":cid,**sc,**rawm})
    return {"raw":raw,"train_df":train_df,"test_df":test_df,"clients_raw":clients,"client_ids":ids,"dq_schema":dq_schema,"dq_reference":reference,"dq_scores":dq_scores,"dq_audit":pd.DataFrame(rows),
            "meta":{"raw_rows":len(raw),"train_rows":len(train_df),"test_rows":len(test_df),"patient_overlap":int(patient_overlap),"n_clients":n_clients},
            "client_train_ids":{cid:df["_row_id"].astype(str).to_numpy() for cid,df in clients.items()},"global_train_ids":train_df["_row_id"].astype(str).to_numpy(),"global_test_ids":test_df["_row_id"].astype(str).to_numpy()}



def build_diabetes_model(input_dim: int, lr: float = 1e-3) -> keras.Model:
    inp = keras.Input(shape=(int(input_dim),), dtype=tf.float32)
    x = layers.Dense(128, activation="relu")(inp)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.20)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.20)(x)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(3, activation="softmax", dtype=tf.float32)(x)
    model = keras.Model(inp, out)
    model.optimizer = keras.optimizers.Adam(learning_rate=float(lr))
    return model



# ======================================================================================
# FULL EXPERIMENT-A REPORTING, CHECKPOINTING, LEDGER, AND STATISTICS
# ======================================================================================
from datetime import datetime, timezone
import re

# =============================================================================
# MAIN REPEATED TRAINING SET
#
# These eight configurations are genuinely distinct and are repeated over all
# five training seeds for mean ± SD / CI reporting.
#
# NOT repeated here:
#   - Great Expectations Centralized
#   - TADP-AA Centralized
#   - Great Expectations Federated
#   - TADP-AA Federated
# Those all-client equivalences are verified independently by the companion
# one-seed equivalence-audit script.
#
# TADP-SDA is retained as a boundary/stress condition but is run once only,
# outside the five-seed headline statistical comparison.
# =============================================================================
MANUSCRIPT_SCENARIOS = [
    "Naïve Centralized",
    "TADP-VR Centralized",
    "Vanilla FedAvg",
    "FedProx",
    "Random-K",
    "DQ-only Federated",
    "Power-of-Choice",
    "TADP-VR Federated",
]
assert len(MANUSCRIPT_SCENARIOS) == 8

BOUNDARY_SCENARIOS = [
    "TADP-SDA Centralized",
    "TADP-SDA Federated",
]
BOUNDARY_SEED = 42



def print_banner(title: str, width: int = 108):
    print("\n" + "=" * width)
    print(title)
    print("=" * width)


def client_partition_table(clients_raw):
    rows = []
    for cid, df in clients_raw.items():
        counts = df["_target"].value_counts().to_dict()
        rows.append({
            "client": cid,
            "records": int(len(df)),
            "class_0_NO": int(counts.get(0, 0)),
            "class_1_GT30": int(counts.get(1, 0)),
            "class_2_LT30": int(counts.get(2, 0)),
        })
    return pd.DataFrame(rows)


def choose_experiment_root(experiment_name, use_drive=True):
    if use_drive:
        try:
            from google.colab import drive
            drive.mount("/content/drive", force_remount=False)
            root = Path("/content/drive/MyDrive/TADP_CHECKPOINTS") / experiment_name
            root.mkdir(parents=True, exist_ok=True)
            print(f"Persistent checkpoint root: {root}")
            return root
        except Exception as exc:
            print(f"Google Drive checkpoint mount unavailable: {exc}")
    root = Path("/content") / experiment_name
    root.mkdir(parents=True, exist_ok=True)
    print(f"Local checkpoint root: {root}")
    return root


def atomic_write_json(obj, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")
    tmp.replace(path)


def load_checkpoint_state(path):
    path = Path(path)
    if not path.exists():
        return {"completed": [], "last_completed": None, "updated_utc": None}
    return json.loads(path.read_text(encoding="utf-8"))


def mark_checkpoint_complete(state_path, key, extra=None):
    state = load_checkpoint_state(state_path)
    completed = list(state.get("completed", []))
    if key not in completed:
        completed.append(key)
    state["completed"] = completed
    state["last_completed"] = key
    state["updated_utc"] = datetime.now(timezone.utc).isoformat()
    if extra:
        state.update(extra)
    atomic_write_json(state, state_path)


def upsert_csv(row, path, key_cols):
    path = Path(path)
    new = pd.DataFrame([row])
    if path.exists():
        old = pd.read_csv(path)
        if not old.empty:
            mask = pd.Series(True, index=old.index)
            for c in key_cols:
                mask &= old[c].astype(str).eq(str(row[c]))
            old = old.loc[~mask].copy()
            new = pd.concat([old, new], ignore_index=True)
    new.to_csv(path, index=False)


def write_scenario_checkpoint(root, run_idx, scenario, result):
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", scenario).strip("_")
    cdir = ensure_dir(root / "scenario_checkpoints" / f"run_{run_idx:02d}")
    payload = {
        "run": int(run_idx),
        "scenario": scenario,
        "completed_utc": datetime.now(timezone.utc).isoformat(),
        "metrics": result["metrics"],
        "runtime_s": float(result["runtime_s"]),
        "optimizer_steps": int(result["total_optimizer_steps"]),
        "communication_mb": float(result.get("communication_mb", 0.0)),
        "ram_peak_mb": float(result.get("ram_peak_mb", 0.0)),
    }
    atomic_write_json(payload, cdir / f"{safe}.json")
    if "round_audit" in result:
        pd.DataFrame(result["round_audit"]).to_csv(
            cdir / f"{safe}_rounds.csv", index=False
        )


def ci95_mean(values):
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan, np.nan
    if len(x) == 1:
        return float(x[0]), float(x[0])
    mean = float(np.mean(x))
    sd = float(np.std(x, ddof=1))
    try:
        from scipy.stats import t
        crit = float(t.ppf(0.975, df=len(x)-1))
    except Exception:
        crit = 1.96
    half = crit * sd / math.sqrt(len(x))
    return mean-half, mean+half


def summarize_runs_with_ci(perf):
    metrics = [
        "accuracy", "precision_macro", "recall_macro", "f1_macro",
        "roc_auc_ovr_macro", "runtime_s", "energy_wh", "energy_kwh",
        "co2_kg", "communication_mb", "ram_peak_mb", "ram_delta_mb",
        "optimizer_steps", "participants", "energy_cost_usd",
        "communication_cost_usd", "total_estimated_cost_usd",
    ]
    rows = []
    for scenario, d in perf.groupby("scenario", sort=False):
        row = {"scenario": scenario, "n_runs": int(len(d))}
        for metric in metrics:
            if metric not in d.columns:
                continue
            vals = pd.to_numeric(d[metric], errors="coerce")
            vals = vals[np.isfinite(vals)]
            row[f"{metric}_mean"] = float(vals.mean()) if len(vals) else np.nan
            row[f"{metric}_sd"] = float(vals.std(ddof=1)) if len(vals)>1 else 0.0
            lo, hi = ci95_mean(vals)
            row[f"{metric}_ci95_low"] = lo
            row[f"{metric}_ci95_high"] = hi
        rows.append(row)
    return pd.DataFrame(rows)


def paired_vr_randomk_statistics(perf):
    vr = perf[perf["scenario"].eq("TADP-VR Federated")].copy()
    rk = perf[perf["scenario"].eq("Random-K")].copy()
    merged = vr.merge(rk, on=["run", "seed"], suffixes=("_vr", "_randomk"), validate="one_to_one")
    rows = []
    for metric in ["accuracy", "f1_macro", "roc_auc_ovr_macro"]:
        a = merged[f"{metric}_vr"].to_numpy(dtype=float)
        b = merged[f"{metric}_randomk"].to_numpy(dtype=float)
        diff = a-b
        mean = float(np.mean(diff))
        sd = float(np.std(diff, ddof=1)) if len(diff)>1 else 0.0
        lo, hi = ci95_mean(diff)
        t_stat=t_p=wil_stat=wil_p=np.nan
        try:
            from scipy.stats import ttest_rel, wilcoxon
            tr = ttest_rel(a,b,nan_policy="omit")
            t_stat, t_p = float(tr.statistic), float(tr.pvalue)
            if np.any(np.abs(diff)>0):
                wr = wilcoxon(a,b)
                wil_stat, wil_p = float(wr.statistic), float(wr.pvalue)
        except Exception:
            pass
        rows.append({
            "metric": metric,
            "n_pairs": len(diff),
            "mean_difference_vr_minus_randomk": mean,
            "sd_difference": sd,
            "ci95_low": lo,
            "ci95_high": hi,
            "cohens_dz": float(mean/sd) if sd>0 else np.nan,
            "paired_t_stat": t_stat,
            "paired_t_p": t_p,
            "wilcoxon_stat": wil_stat,
            "wilcoxon_p": wil_p,
            "vr_wins": int(np.sum(diff>0)),
            "ties": int(np.sum(np.isclose(diff,0))),
            "vr_losses": int(np.sum(diff<0)),
        })
    return pd.DataFrame(rows)


# ======================================================================================
# TADP EXPERIMENT G — ADVERSARIAL DIAGNOSTIC PROBE
# ======================================================================================
# PURPOSE
# -------
# This is a deliberately scoped DIAGNOSTIC experiment for the manuscript response to
# the reviewer concern about malicious contributors. It does NOT claim that TADP is a
# poisoning/backdoor defense and it does NOT evaluate arbitrary Byzantine model-update
# replacement, server compromise, collusion, or ledger tampering. Those runtime-security
# threats remain within the TADP-Sec extension.
#
# Threat model tested here:
#   * a malicious contributor controls its own local TRAIN data and therefore the local
#     update produced from those data;
#   * it cannot alter other clients, the server, the frozen global holdout, or the ledger;
#   * documentary evidence is assumed to be verified before normal TADP scoring;
#   * PROVENANCE_INFLATION is therefore an explicit counterfactual verification-bypass
#     diagnostic in which forged documentary claims are allowed through verification;
#     all documentary factors are then set to their maximum score, while Data Quality
#     remains machine-measured from the attacked TRAIN data.
#
# Attacks:
#   1) LABEL_FLIP: cyclic valid-label flipping on a fixed fraction of each malicious
#      client's TRAIN records. This preserves the valid label domain by construction.
#   2) BACKDOOR: a rare but schema-valid categorical trigger (derived from TRAIN only)
#      is inserted into a fixed fraction of malicious TRAIN records and their labels are
#      changed to a fixed target class. Attack success rate (ASR) is measured only on a
#      triggered copy of the held-out TEST set after training; it never affects training,
#      governance, client selection, or tuning.
#   3) PROVENANCE_INFLATION: a deliberately poor local dataset is paired with maximally
#      favorable forged documentary scores under the counterfactual assumption that
#      documentary verification has been bypassed. This tests whether machine-measured
#      DQ plus the 2.5 dimension floor can still prevent documentary-score compensation.
#
# Contamination prevalence is defined over the K=10 client population. Governance-only
# diagnostics are evaluated at 10%, 20%, and 30% (1, 2, and 3 malicious clients). To keep
# the reviewer-response experiment compact, model training is performed only at the 20%
# contamination level. Malicious clients are selected only from the CLEAN TADP-VR eligible
# cohort so the probe asks whether an otherwise eligible contributor can evade the gate.
#
# Training comparisons (20% contamination only):
#   * ALL_CLIENT: attacked FedAvg with all 10 clients (no TADP participation gate).
#   * TADP_GATE: attacked FedAvg using the TADP-VR cohort recomputed after the attack.
# Clean Accuracy/F1/AUC references are reused from the already completed final Experiment A1 Option-A
# for the same seeds, split, model, rounds, and participation regimes. Therefore no clean
# models are retrained here. Backdoor ASR is reported as an absolute diagnostic because the
# Experiment A clean models were not retained for triggered-test evaluation.
# ======================================================================================

import copy

EXPERIMENT_VERSION = "TADP-G-FINAL-A1A2-28F-HPSREVIEW-OPTIONA-ADVERSARIAL-K10-3SEED-4ROUND"
TRAINING_RUN_SEEDS=[42,142,242]
NUM_CLIENTS=10
NUM_ROUNDS_FL=4
LOCAL_EPOCHS=1
BATCH_SIZE=64
LEARNING_RATE=1e-3
DIRICHLET_ALPHA=1.0
GLOBAL_SPLIT_SEED=7001
CLIENT_PARTITION_SEED=7101
FROZEN_EVIDENCE_ASSIGNMENT_SEED=1042
ATTACK_CLIENT_SELECTION_SEED=9042
CONTAMINATION_LEVELS=[0.10,0.20,0.30]
TRAINING_CONTAMINATION_LEVEL=0.20
LABEL_FLIP_RECORD_FRACTION=0.20
BACKDOOR_RECORD_FRACTION=0.10
PROVENANCE_BAD_RECORD_FRACTION=0.45
PROVENANCE_DUPLICATE_RECORD_FRACTION=0.30
BACKDOOR_TARGET_CLASS=2
BACKDOOR_TRIGGER_FEATURES=3
DOMAIN="healthcare"
USE_GOOGLE_DRIVE_CHECKPOINTS=True
EXPERIMENT_ROOT=choose_experiment_root("TADP_EXPERIMENT_G_"+EXPERIMENT_VERSION,use_drive=USE_GOOGLE_DRIVE_CHECKPOINTS)
CHECKPOINT_STATE=EXPERIMENT_ROOT/"checkpoint_state.json"
PERF_CHECKPOINT=EXPERIMENT_ROOT/"adversarial_performance_checkpoint.csv"
GOV_AUDIT_PATH=EXPERIMENT_ROOT/"adversarial_governance_audit.csv"
ATTACK_AUDIT_PATH=EXPERIMENT_ROOT/"attack_data_audit.csv"
ATTACKS=["LABEL_FLIP","BACKDOOR","PROVENANCE_INFLATION"]
GATE_MODES=["ALL_CLIENT","TADP_GATE"]
# Exact final A1 seed-matched clean references (same split/partition/training seeds).
EXPERIMENT_A_CLEAN_REFERENCE={
42:{"ALL_CLIENT":{"accuracy":0.462654,"f1_macro":0.352802,"roc_auc_ovr_macro":0.631308},"TADP_GATE":{"accuracy":0.474100,"f1_macro":0.309793,"roc_auc_ovr_macro":0.629268}},
142:{"ALL_CLIENT":{"accuracy":0.462363,"f1_macro":0.361459,"roc_auc_ovr_macro":0.636923},"TADP_GATE":{"accuracy":0.456931,"f1_macro":0.311004,"roc_auc_ovr_macro":0.632520}},
242:{"ALL_CLIENT":{"accuracy":0.504899,"f1_macro":0.372336,"roc_auc_ovr_macro":0.632819},"TADP_GATE":{"accuracy":0.450238,"f1_macro":0.304264,"roc_auc_ovr_macro":0.625320}},
}

def clean_reference_dataframe():
    rows=[]
    for seed,modes in EXPERIMENT_A_CLEAN_REFERENCE.items():
        for mode,metrics in modes.items(): rows.append({"seed":seed,"gate_mode":mode,"clean_accuracy":metrics["accuracy"],"clean_f1_macro":metrics["f1_macro"],"clean_roc_auc_ovr_macro":metrics["roc_auc_ovr_macro"],"clean_reference_source":"Final Experiment A1 Option-A run"})
    return pd.DataFrame(rows)



def _stable_client_seed(base_seed: int, cid: str, salt: int = 0) -> int:
    digest = hashlib.sha256(str(cid).encode("utf-8")).hexdigest()
    return int(base_seed + salt + (int(digest[:8], 16) % 1000000))


def _target_string(class_id: int) -> str:
    mapping = {0: "NO", 1: ">30", 2: "<30"}
    return mapping[int(class_id)]


def copy_documentary_evidence(evidence):
    return copy.deepcopy(evidence)


def derive_backdoor_trigger(
    train_df: pd.DataFrame,
    preprocessor: Any,
    n_features: int = BACKDOOR_TRIGGER_FEATURES,
) -> Dict[str, str]:
    """
    Derive a fixed rare-but-valid categorical conjunction from TRAIN only.

    The trigger uses existing category values, so it does not introduce a new schema
    token. No TEST information is consulted. Preferred low-cardinality clinical fields
    are used when available; otherwise a deterministic categorical fallback is used.
    """
    preferred = [
        "A1Cresult",
        "max_glu_serum",
        "change",
        "diabetesMed",
        "insulin",
        "metformin",
        "glipizide",
        "glyburide",
    ]

    cat_cols = [c for c in preprocessor.categorical_cols if c in train_df.columns]
    ordered = [c for c in preferred if c in cat_cols]

    fallback = []
    for c in cat_cols:
        if c in ordered:
            continue
        s = train_df[c].astype("string").dropna().astype(str)
        nunique = int(s.nunique())
        if 2 <= nunique <= 20:
            fallback.append(c)
    ordered.extend(sorted(fallback))

    trigger = {}
    n_train = len(train_df)
    min_support = max(20, int(round(0.001 * n_train)))

    for c in ordered:
        s = train_df[c].astype("string").dropna().astype(str)
        vc = s.value_counts()
        vc = vc[(vc.index != "__MISSING__") & (vc >= min_support)]
        if len(vc) < 2:
            continue
        # Rare valid value, with deterministic lexical tie-break.
        min_count = int(vc.min())
        candidates = sorted([str(v) for v in vc[vc.eq(min_count)].index.tolist()])
        trigger[c] = candidates[0]
        if len(trigger) >= int(n_features):
            break

    if len(trigger) < int(n_features):
        raise RuntimeError(
            f"Could not derive {n_features} schema-valid categorical trigger fields "
            f"from TRAIN. Derived={trigger}"
        )
    return trigger


def trigger_natural_prevalence(df: pd.DataFrame, trigger: Dict[str, str]) -> float:
    if not trigger:
        return 0.0
    mask = np.ones(len(df), dtype=bool)
    for c, v in trigger.items():
        mask &= df[c].astype("string").fillna("__MISSING__").astype(str).eq(str(v)).to_numpy()
    return float(mask.mean()) if len(mask) else 0.0


def apply_label_flip(
    df: pd.DataFrame,
    fraction: float,
    seed: int,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    out = df.copy(deep=True)
    rng = np.random.default_rng(int(seed))
    n = len(out)
    n_poison = int(max(1, round(float(fraction) * n)))
    idx = rng.choice(n, size=min(n_poison, n), replace=False)
    old_y = out.iloc[idx]["_target"].to_numpy(dtype=np.int32)
    new_y = (old_y + 1) % 3
    out.loc[out.index[idx], "_target"] = new_y
    if TARGET in out.columns:
        out.loc[out.index[idx], TARGET] = [_target_string(y) for y in new_y]
    return out, {
        "records_total": int(n),
        "records_poisoned": int(len(idx)),
        "record_fraction_realized": float(len(idx) / max(1, n)),
        "attack_detail": "cyclic valid-label flip: 0->1, 1->2, 2->0",
    }


def apply_backdoor(
    df: pd.DataFrame,
    trigger: Dict[str, str],
    target_class: int,
    fraction: float,
    seed: int,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    out = df.copy(deep=True)
    rng = np.random.default_rng(int(seed))
    candidate_pos = np.where(out["_target"].to_numpy(dtype=np.int32) != int(target_class))[0]
    if len(candidate_pos) == 0:
        raise RuntimeError("Backdoor attack has no non-target local records to poison.")
    n_target = int(max(1, round(float(fraction) * len(out))))
    n_poison = min(n_target, len(candidate_pos))
    chosen = rng.choice(candidate_pos, size=n_poison, replace=False)
    chosen_index = out.index[chosen]
    for c, v in trigger.items():
        out.loc[chosen_index, c] = str(v)
    out.loc[chosen_index, "_target"] = int(target_class)
    if TARGET in out.columns:
        out.loc[chosen_index, TARGET] = _target_string(target_class)
    return out, {
        "records_total": int(len(out)),
        "records_poisoned": int(n_poison),
        "record_fraction_realized": float(n_poison / max(1, len(out))),
        "attack_detail": json.dumps({
            "target_class": int(target_class),
            "trigger": trigger,
        }, sort_keys=True),
    }


def apply_severe_quality_degradation(
    df: pd.DataFrame,
    preprocessor: Any,
    bad_fraction: float,
    duplicate_fraction: float,
    seed: int,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """
    Fixed severe poor-data stressor used only for PROVENANCE_INFLATION.

    Row count and labels are preserved. A disjoint subset is made feature-duplicate,
    while another subset receives malformed numeric values and extensive missing
    categorical values. The corruption is fixed before governance and never tuned on
    TEST performance.
    """
    out = df.copy(deep=True)
    rng = np.random.default_rng(int(seed))
    n = len(out)
    if n < 5:
        raise RuntimeError("Client shard too small for quality-degradation probe.")

    n_dup = min(int(round(float(duplicate_fraction) * n)), n - 2)
    all_pos = np.arange(n)
    dup_pos = rng.choice(all_pos, size=max(1, n_dup), replace=False)
    remaining = np.setdiff1d(all_pos, dup_pos, assume_unique=False)
    n_bad = min(int(round(float(bad_fraction) * n)), len(remaining))
    bad_pos = rng.choice(remaining, size=max(1, n_bad), replace=False)

    feature_cols = [c for c in preprocessor.feature_cols if c in out.columns]
    donor_candidates = np.setdiff1d(remaining, bad_pos, assume_unique=False)
    donor_pos = int(donor_candidates[0] if len(donor_candidates) else remaining[0])
    donor = out.iloc[donor_pos][feature_cols].copy()

    # Create exact duplicate feature patterns while preserving unique IDs/row IDs.
    for c in feature_cols:
        out.loc[out.index[dup_pos], c] = donor[c]

    # Malformed numeric values: fixed clean preprocessor later coerces them to NaN.
    for c in preprocessor.numeric_cols:
        if c in out.columns:
            out.loc[out.index[bad_pos], c] = "__INVALID_NUMERIC__"

    # Extensive categorical missingness on the same poor-quality rows.
    cat_cols = [c for c in preprocessor.categorical_cols if c in out.columns]
    n_cat_bad = int(max(1, math.ceil(0.70 * len(cat_cols)))) if cat_cols else 0
    for c in cat_cols[:n_cat_bad]:
        out.loc[out.index[bad_pos], c] = np.nan

    return out, {
        "records_total": int(n),
        "records_duplicate_pattern": int(len(dup_pos)),
        "records_malformed_or_missing": int(len(bad_pos)),
        "record_fraction_realized": float((len(dup_pos) + len(bad_pos)) / max(1, n)),
        "attack_detail": (
            f"feature-duplicate fraction={len(dup_pos)/max(1,n):.3f}; "
            f"malformed/missing fraction={len(bad_pos)/max(1,n):.3f}; "
            "labels preserved"
        ),
    }


def inflate_documentary_scores_verification_bypass(evidence, malicious_clients, domain=DOMAIN):
    """Counterfactual forged-evidence diagnostic: inflate all documentary factors to 5.
    This assumes false claims have bypassed documentary verification; machine-measured DQ is never modified here.
    """
    out=copy_documentary_evidence(evidence)
    for cid in malicious_clients:
        for d in DOCUMENTARY_DIMS:
            for f in FACTOR_NAMES[d]: out[cid][d][f]=5.0
    return out



def recompute_dq_and_arrays(client_frames, preprocessor, reference):
    dq_scores={}; rows=[]
    for cid,frame in client_frames.items():
        scores,raw=tabular_dq_scores(frame,preprocessor,reference); dq_scores[str(cid)]=scores; rows.append({"client":str(cid),**scores,**raw})
    return dq_scores,pd.DataFrame(rows),{}



def build_triggered_test(
    test_df: pd.DataFrame,
    preprocessor: Any,
    trigger: Dict[str, str],
    target_class: int,
) -> Tuple[np.ndarray, np.ndarray]:
    work = test_df.loc[test_df["_target"].ne(int(target_class))].copy(deep=True)
    if work.empty:
        raise RuntimeError("No non-target TEST rows available for backdoor ASR evaluation.")
    y_true = work["_target"].to_numpy(dtype=np.int32)
    for c, v in trigger.items():
        work[c] = str(v)
    X = preprocessor.transform(work)
    return X, y_true


def evaluate_backdoor_asr(
    model: keras.Model,
    X_triggered: np.ndarray,
    target_class: int,
) -> float:
    p = model.predict(X_triggered, batch_size=512, verbose=0)
    pred = np.argmax(p, axis=1)
    return float(np.mean(pred == int(target_class)))


def make_attack_condition(
    clean_clients_raw: Dict[str, pd.DataFrame],
    clean_evidence: Dict[str, Dict[str, Dict[str, float]]],
    malicious_clients: List[str],
    attack: str,
    preprocessor: Any,
    trigger: Dict[str, str],
    condition_seed: int,
) -> Tuple[
    Dict[str, pd.DataFrame],
    Dict[str, Dict[str, Dict[str, float]]],
    pd.DataFrame,
]:
    frames = {cid: df.copy(deep=True) for cid, df in clean_clients_raw.items()}
    evidence = copy_documentary_evidence(clean_evidence)
    audit_rows = []

    for cid in malicious_clients:
        seed = _stable_client_seed(condition_seed, cid, salt=1700)
        if attack == "LABEL_FLIP":
            frames[cid], audit = apply_label_flip(
                frames[cid], LABEL_FLIP_RECORD_FRACTION, seed
            )
        elif attack == "BACKDOOR":
            frames[cid], audit = apply_backdoor(
                frames[cid], trigger, BACKDOOR_TARGET_CLASS,
                BACKDOOR_RECORD_FRACTION, seed,
            )
        elif attack == "PROVENANCE_INFLATION":
            frames[cid], audit = apply_severe_quality_degradation(
                frames[cid], preprocessor,
                PROVENANCE_BAD_RECORD_FRACTION,
                PROVENANCE_DUPLICATE_RECORD_FRACTION,
                seed,
            )
        else:
            raise ValueError(f"Unknown attack={attack!r}")

        audit_rows.append({
            "attack": attack,
            "client": str(cid),
            "malicious": True,
            "condition_seed": int(condition_seed),
            **audit,
        })

    if attack == "PROVENANCE_INFLATION":
        evidence = inflate_documentary_scores_verification_bypass(
            evidence, malicious_clients, domain=DOMAIN
        )

    return frames, evidence, pd.DataFrame(audit_rows)


def build_governance_attack_audit(gov, malicious_clients, attack, prevalence, condition_seed):
    out=gov.copy(); out["attack"]=attack; out["prevalence"]=prevalence; out["condition_seed"]=condition_seed; out["malicious"]=out.client.astype(str).isin(set(map(str,malicious_clients)))
    keep=["attack","prevalence","condition_seed","client","malicious","evidence_validity_pass","evidence_nonpositive","hps","hps_review","hps_review_midpoint","dimension_floor_failures","dim1_score_0_5","dim2_score_0_5","dim3_score_0_5","dim4_score_0_5","dim5_score_0_5","dim6_score_0_5","decision_path","final_action","status","reason"]
    return out[[c for c in keep if c in out.columns]].copy()



def governance_condition_stats(gov, malicious_clients):
    malicious=set(map(str,malicious_clients)); g=gov.copy(); g["client"]=g.client.astype(str); gm=g[g.client.isin(malicious)]; gh=g[~g.client.isin(malicious)]; admitted=gm.final_action.eq("ACCEPT")
    return {"adversary_admitted_count":int(admitted.sum()),"adversary_total_count":len(gm),"adversary_admission_rate":float(admitted.mean()) if len(gm) else np.nan,
            "malicious_hps_mean":float(gm.hps.mean()) if len(gm) else np.nan,"honest_hps_mean":float(gh.hps.mean()) if len(gh) else np.nan,"malicious_dq_mean":float(gm.dim2_score_0_5.mean()) if len(gm) else np.nan,"honest_dq_mean":float(gh.dim2_score_0_5.mean()) if len(gh) else np.nan,
            "evidence_validity_rejects_malicious":int(gm.status.eq("AUTO_REJECTED_EVIDENCE_VALIDITY").sum()),"dimension_floor_rejects_malicious":int(gm.status.eq("AUTO_REJECTED_DIMENSION_FLOOR").sum()),"low_hps_rejects_malicious":int(gm.status.eq("AUTO_REJECTED_LOW_HPS").sum()),"hps_review_rejects_malicious":int(gm.status.eq("AUTO_REJECTED_HPS_REVIEW").sum())}



def run_fedavg_condition(
    selected_clients: List[str],
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    build_model_fn,
    initial_weights: List[np.ndarray],
    X_test: np.ndarray,
    y_test: np.ndarray,
    class_weights: Dict[int, float],
    run_seed: int,
    scenario_label: str,
    overall_idx: int,
    overall_total: int,
    train_monitor,
) -> Dict[str, Any]:
    if not selected_clients:
        raise RuntimeError(f"{scenario_label}: no clients selected for training.")
    selected_per_round = [list(selected_clients) for _ in range(NUM_ROUNDS_FL)]
    return federated_train(
        build_model_fn=build_model_fn,
        initial_weights=initial_weights,
        selected_per_round=selected_per_round,
        client_arrays=client_arrays,
        X_test=X_test,
        y_test=y_test,
        n_classes=3,
        batch_size=BATCH_SIZE,
        local_epochs=LOCAL_EPOCHS,
        class_weights=class_weights,
        run_seed=int(run_seed),
        exact_step_maps=None,
        equal_weight=False,
        fedprox_mu=0.0,
        train_monitor=train_monitor,
        progress_context={
            "scenario": scenario_label,
            "run_idx": 1,
            "run_total": 1,
            "scenario_idx": int(overall_idx),
            "scenario_total": int(overall_total),
            "overall_idx": int(overall_idx),
            "overall_total": int(overall_total),
        },
    )


def append_table(df: pd.DataFrame, path: Path):
    if df is None or df.empty:
        return
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, mode="a", header=not path.exists(), index=False)


def _scenario_key(run_idx, attack, prevalence, gate_mode):
    return (
        f"run={int(run_idx)}|attack={attack}|prev={float(prevalence):.2f}|"
        f"gate={gate_mode}"
    )


def summarize_adversarial_results(perf: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    p = perf.copy()
    clean = clean_reference_dataframe()
    attacked = p.merge(
        clean,
        on=["seed", "gate_mode"],
        how="left",
        validate="many_to_one",
    )
    if attacked[["clean_accuracy", "clean_f1_macro", "clean_roc_auc_ovr_macro"]].isna().any().any():
        raise RuntimeError("Missing Experiment A clean reference for one or more training rows.")

    attacked["accuracy_degradation"] = attacked["clean_accuracy"] - attacked["accuracy"]
    attacked["f1_degradation"] = attacked["clean_f1_macro"] - attacked["f1_macro"]
    attacked["auc_degradation"] = attacked["clean_roc_auc_ovr_macro"] - attacked["roc_auc_ovr_macro"]

    metrics = [
        "adversary_admission_rate", "malicious_hps_mean", "honest_hps_mean",
        "malicious_dq_mean", "honest_dq_mean", "accuracy", "f1_macro",
        "roc_auc_ovr_macro", "accuracy_degradation", "f1_degradation",
        "auc_degradation", "backdoor_asr", "selected_client_count",
        "optimizer_steps", "runtime_s", "communication_mb",
    ]
    rows = []
    for keys, d in attacked.groupby(["attack", "prevalence", "gate_mode"], sort=False):
        attack, prev, gate_mode = keys
        row = {
            "attack": attack,
            "prevalence": float(prev),
            "gate_mode": gate_mode,
            "n_runs": int(len(d)),
        }
        for m in metrics:
            vals = pd.to_numeric(d[m], errors="coerce")
            vals = vals[np.isfinite(vals)]
            row[f"{m}_mean"] = float(vals.mean()) if len(vals) else np.nan
            row[f"{m}_sd"] = float(vals.std(ddof=1)) if len(vals) > 1 else 0.0
        rows.append(row)
    return attacked, pd.DataFrame(rows)

def write_adversarial_figures(
    summary: pd.DataFrame,
    governance_summary: pd.DataFrame,
    out_dir: Path,
):
    try:
        import matplotlib.pyplot as plt
    except Exception as exc:
        print(f"Figure generation skipped: {exc}")
        return

    # Figure 1: governance-only admission rate across 10%, 20%, and 30%.
    d = governance_summary.copy()
    if not d.empty:
        fig, ax = plt.subplots(figsize=(8, 5))
        for attack, g in d.groupby("attack", sort=False):
            g = g.sort_values("prevalence")
            ax.plot(
                100.0 * g["prevalence"].to_numpy(dtype=float),
                100.0 * g["adversary_admission_rate"].to_numpy(dtype=float),
                marker="o",
                label=attack.replace("_", " ").title(),
            )
        ax.set_xlabel("Malicious clients (% of K=10)")
        ax.set_ylabel("Adversarial client admission rate (%)")
        ax.set_ylim(-5, 105)
        ax.grid(alpha=0.25)
        ax.legend()
        fig.tight_layout()
        fig.savefig(Path(out_dir) / "Figure_Adversarial_Admission_Rate_GovernanceOnly.png", dpi=220)
        plt.close(fig)

    # Figure 2: macro-F1 degradation at the single trained contamination level (20%).
    if not summary.empty:
        work = summary.copy()
        labels = []
        values = []
        for _, r in work.iterrows():
            labels.append(
                f"{str(r['attack']).replace('_',' ').title()}\n{str(r['gate_mode']).replace('_',' ').title()}"
            )
            values.append(float(r["f1_degradation_mean"]))
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.bar(np.arange(len(values)), values)
        ax.axhline(0.0, linewidth=1.0)
        ax.set_xticks(np.arange(len(values)))
        ax.set_xticklabels(labels, rotation=20, ha="right")
        ax.set_ylabel("Macro-F1 degradation vs Experiment A clean reference")
        ax.set_title("Adversarial diagnostic at 20% malicious clients")
        fig.tight_layout()
        fig.savefig(Path(out_dir) / "Figure_Adversarial_F1_Degradation_20pct.png", dpi=220)
        plt.close(fig)

    # Figure 3: absolute backdoor ASR at 20%.
    b = summary[summary["attack"].eq("BACKDOOR")].copy()
    if not b.empty:
        fig, ax = plt.subplots(figsize=(7, 5))
        labels = b["gate_mode"].str.replace("_", " ", regex=False).str.title().tolist()
        vals = 100.0 * b["backdoor_asr_mean"].to_numpy(dtype=float)
        ax.bar(np.arange(len(vals)), vals)
        ax.set_xticks(np.arange(len(vals)))
        ax.set_xticklabels(labels)
        ax.set_ylabel("Backdoor attack success rate (%)")
        ax.set_ylim(0, 100)
        ax.set_title("Backdoor diagnostic at 20% malicious clients")
        fig.tight_layout()
        fig.savefig(Path(out_dir) / "Figure_Backdoor_ASR_20pct.png", dpi=220)
        plt.close(fig)

def main():
    csv_path=locate_diabetes_csv(); print_banner(EXPERIMENT_VERSION)
    print("Diagnostic only: not a general poisoning/backdoor/Byzantine defense evaluation.")
    data=prepare_diabetes_no_leakage(csv_path,GLOBAL_SPLIT_SEED,CLIENT_PARTITION_SEED,NUM_CLIENTS,DIRICHLET_ALPHA)
    leakage=write_leakage_audit(EXPERIMENT_ROOT,data["global_train_ids"],data["global_test_ids"],data["client_train_ids"],extra={"patient_overlap":data["meta"]["patient_overlap"],"global_holdout_before_client_partition":True,"optionA_post_selection_model_preprocessing":True,"dq_governance_raw_train_only":True,"test_used_for_final_evaluation_only":True})
    client_ids=list(map(str,data["client_ids"])); clean_evidence,evidence_df=generate_controlled_documentary_evidence(client_ids,FROZEN_EVIDENCE_ASSIGNMENT_SEED,domain=DOMAIN); evidence_df.to_csv(EXPERIMENT_ROOT/"clean_controlled_documentary_evidence.csv",index=False)
    clean_gov=build_tadp_governance(client_ids,clean_evidence,data["dq_scores"],0,FROZEN_EVIDENCE_ASSIGNMENT_SEED,DOMAIN); clean_gov.to_csv(EXPERIMENT_ROOT/"clean_governance.csv",index=False); clean_vr=accepted_tadp_vr(clean_gov)
    expected=["A","B","G","H","I","J"]
    if clean_vr!=expected: raise RuntimeError(f"Clean final-policy cohort mismatch: expected {expected}, got {clean_vr}")
    rng=np.random.default_rng(ATTACK_CLIENT_SELECTION_SEED); order=list(rng.permutation(clean_vr)); attacker_sets={float(p):list(map(str,order[:max(1,int(round(p*NUM_CLIENTS))) ])) for p in CONTAMINATION_LEVELS}
    trigger=derive_backdoor_trigger(data["train_df"],data["dq_schema"],BACKDOOR_TRIGGER_FEATURES); natural_train=trigger_natural_prevalence(data["train_df"],trigger); natural_test=trigger_natural_prevalence(data["test_df"],trigger)
    design={"experiment_version":EXPERIMENT_VERSION,"purpose":"limited adversarial boundary diagnostic","training_seeds":TRAINING_RUN_SEEDS,"K":NUM_CLIENTS,"rounds":NUM_ROUNDS_FL,"governance_only_contamination_levels":CONTAMINATION_LEVELS,"training_contamination_level":TRAINING_CONTAMINATION_LEVEL,
            "final_policy":{"evidence_validity":"all 28 factors present finite >0","dimension_floor":DIMENSION_MIN_FLOOR,"hps_lower":GOOD_CUT,"hps_upper":HIGH_CUT,"hps_review_threshold":REVIEW_ACCEPT_THRESHOLD,"critical_review_dimensions":CRITICAL_REVIEW_DIMS,"critical_review_weights":CRITICAL_REVIEW_WEIGHTS},
            "optionA_post_selection_preprocessing":True,"provenance_inflation_scope":"all documentary factors set to 5 under counterfactual verification-bypass diagnostic; machine DQ unchanged","backdoor_trigger":trigger,"clean_reference_source":"final Experiment A1 Option-A"}
    atomic_write_json(design,EXPERIMENT_ROOT/"threat_model_and_experiment_design.json"); clean_reference_dataframe().to_csv(EXPERIMENT_ROOT/"experiment_A1_final_clean_reference_metrics.csv",index=False)
    conditions={}; gov_audits=[]; attack_audits=[]
    for ai,attack in enumerate(ATTACKS):
        for prevalence in CONTAMINATION_LEVELS:
            malicious=attacker_sets[float(prevalence)]; condition_seed=ATTACK_CLIENT_SELECTION_SEED+10000*(ai+1)+int(round(100*prevalence))
            frames,evidence,attack_audit=make_attack_condition(data["clients_raw"],clean_evidence,malicious,attack,data["dq_schema"],trigger,condition_seed)
            dq_scores,dq_audit,_=recompute_dq_and_arrays(frames,data["dq_schema"],data["dq_reference"]); gov=build_tadp_governance(client_ids,evidence,dq_scores,0,FROZEN_EVIDENCE_ASSIGNMENT_SEED,DOMAIN); accepted=accepted_tadp_vr(gov); stats=governance_condition_stats(gov,malicious)
            conditions[(attack,float(prevalence))]={"frames":frames,"evidence":evidence,"dq_scores":dq_scores,"dq_audit":dq_audit,"gov":gov,"accepted":accepted,"malicious_clients":malicious,"gov_stats":stats,"condition_seed":condition_seed}
            gov_audits.append(build_governance_attack_audit(gov,malicious,attack,prevalence,condition_seed));
            if not attack_audit.empty: attack_audits.append(attack_audit.assign(prevalence=float(prevalence)))
    pd.concat(gov_audits,ignore_index=True).to_csv(GOV_AUDIT_PATH,index=False)
    if attack_audits: pd.concat(attack_audits,ignore_index=True).to_csv(ATTACK_AUDIT_PATH,index=False)
    # Precompute Option-A training views only for the trained 20% conditions.
    train_views={}; prep_rows=[]
    for attack in ATTACKS:
        cond=conditions[(attack,float(TRAINING_CONTAMINATION_LEVEL))]
        for mode in GATE_MODES:
            selected=client_ids if mode=="ALL_CLIENT" else list(cond["accepted"])
            if not selected: continue
            view=build_post_selection_attack_view(cond["frames"],data["test_df"],selected); view["X_triggered_test"],view["y_triggered_true"]=build_triggered_test(data["test_df"],view["preprocessor"],trigger,BACKDOOR_TARGET_CLASS); train_views[(attack,mode)]=view
            prep_rows.append({"attack":attack,"gate_mode":mode,"selected_clients":";".join(selected),"selected_k":len(selected),"input_dim":view["input_dim"],"preprocessor_sha256":view["preprocessor_sha256"],"class_weights_json":json.dumps(view["class_weights"],sort_keys=True)})
    pd.DataFrame(prep_rows).to_csv(EXPERIMENT_ROOT/"scenario_preprocessing_audit.csv",index=False)
    total=len(TRAINING_RUN_SEEDS)*len(train_views); state=load_checkpoint_state(CHECKPOINT_STATE); completed=set(state.get("completed",[])); counter=0
    for run_idx,seed in enumerate(TRAINING_RUN_SEEDS,start=1):
        for attack in ATTACKS:
            cond=conditions[(attack,float(TRAINING_CONTAMINATION_LEVEL))]
            for mode in GATE_MODES:
                counter+=1; key=_scenario_key(run_idx,attack,TRAINING_CONTAMINATION_LEVEL,mode)
                if key in completed: continue
                view=train_views[(attack,mode)]; seed_everything(seed); build_model_fn=lambda dim=view["input_dim"]:build_diabetes_model(dim,LEARNING_RATE); m=build_model_fn(); init=[np.array(w,copy=True) for w in m.get_weights()]; h=sha256_weights(init); del m; tf.keras.backend.clear_session()
                monitor=build_train_monitor_subset(view["client_arrays"],seed=99117+1000*ATTACKS.index(attack)); scenario=f"{attack} | prevalence={TRAINING_CONTAMINATION_LEVEL:.0%} | {mode}"
                result=run_fedavg_condition(view["selected_clients"],view["client_arrays"],build_model_fn,init,view["X_test"],view["y_test"],view["class_weights"],seed,scenario,counter,total,monitor)
                asr=evaluate_backdoor_asr(result["model"],view["X_triggered_test"],BACKDOOR_TARGET_CLASS); row=result_row(run_idx,seed,scenario,result,h)
                row.update({"attack":attack,"prevalence":TRAINING_CONTAMINATION_LEVEL,"gate_mode":mode,"malicious_clients":";".join(cond["malicious_clients"]),**cond["gov_stats"],"selected_client_count":len(view["selected_clients"]),"selected_clients":";".join(view["selected_clients"]),"backdoor_asr":asr,"condition_seed":cond["condition_seed"],"clean_reference_reused":True,"clean_reference_experiment":"TADP-A1-FINAL-OPTIONA","input_dim":view["input_dim"],"preprocessor_sha256":view["preprocessor_sha256"],"preprocessing_fit_client_ids":";".join(view["selected_clients"])})
                upsert_csv(row,PERF_CHECKPOINT,["run","attack","prevalence","gate_mode"]); mark_checkpoint_complete(CHECKPOINT_STATE,key); completed.add(key); del result["model"]; tf.keras.backend.clear_session(); gc.collect()
    perf=pd.read_csv(PERF_CHECKPOINT); detail,summary=summarize_adversarial_results(perf); perf.to_csv(EXPERIMENT_ROOT/"adversarial_performance_all_runs.csv",index=False); detail.to_csv(EXPERIMENT_ROOT/"adversarial_attacked_runs_with_degradation.csv",index=False); summary.to_csv(EXPERIMENT_ROOT/"adversarial_summary_mean_sd.csv",index=False)
    gov_summary=pd.DataFrame([{"attack":a,"prevalence":p,"malicious_clients":";".join(c["malicious_clients"]),"tadp_admitted_clients":";".join(c["accepted"]),"tadp_admitted_count":len(c["accepted"]),**c["gov_stats"]} for (a,p),c in conditions.items()]); gov_summary.to_csv(EXPERIMENT_ROOT/"adversarial_governance_summary.csv",index=False); write_adversarial_figures(summary,gov_summary,EXPERIMENT_ROOT)
    print_banner("INTERPRETATION GUARDRAILS"); print("TADP is a pre-training governance gate, not a general adversarial defense."); print("Provenance inflation assumes documentary verification was bypassed for the forged claims."); print("Backdoor ASR is absolute because clean A1 model objects were not retained.")
    return EXPERIMENT_ROOT



if __name__ == "__main__":
    finished_root = main()
    package_and_download_results(finished_root, EXPERIMENT_VERSION)


📤 Please upload the diabetes_130US.csv file:


Saving diabetes_130US.csv to diabetes_130US (1).csv
✅ Uploaded: diabetes_130US (1).csv (19159383 bytes)
✅ Saved to cache: ./data_cache/diabetes_130US.csv
✅ Verified: ./data_cache/diabetes_130US.csv exists (19159383 bytes)
Google Drive checkpoint mount unavailable: Error: credential propagation was unsuccessful
Local checkpoint root: /content/TADP_EXPERIMENT_G_TADP-G-FINAL-A1A2-28F-HPSREVIEW-OPTIONA-ADVERSARIAL-K10-3SEED-4ROUND

TADP-G-FINAL-A1A2-28F-HPSREVIEW-OPTIONA-ADVERSARIAL-K10-3SEED-4ROUND
Diagnostic only: not a general poisoning/backdoor/Byzantine defense evaluation.


/tmp/ipykernel_676/3982744915.py:2915: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '__INVALID_NUMERIC__' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  out.loc[out.index[bad_pos], c] = "__INVALID_NUMERIC__"
/tmp/ipykernel_676/3982744915.py:2915: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '__INVALID_NUMERIC__' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  out.loc[out.index[bad_pos], c] = "__INVALID_NUMERIC__"
/tmp/ipykernel_676/3982744915.py:2915: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '__INVALID_NUMERIC__' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  out.loc[out.index[bad_pos], c] = "__INVALID_NUMERIC__"
/


INTERPRETATION GUARDRAILS
TADP is a pre-training governance gate, not a general adversarial defense.
Provenance inflation assumes documentary verification was bypassed for the forged claims.
Backdoor ASR is absolute because clean A1 model objects were not retained.

TADP-G-FINAL-A1A2-28F-HPSREVIEW-OPTIONA-ADVERSARIAL-K10-3SEED-4ROUND COMPLETE
Results ZIP: /content/TADP_EXPERIMENT_G_TADP-G-FINAL-A1A2-28F-HPSREVIEW-OPTIONA-ADVERSARIAL-K10-3SEED-4ROUND_RESULTS.zip
ZIP size: 0.19 MB
Starting automatic download to your laptop...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>